# KMTNet $B$-band calibration — full-program batch (v37)

### Files and credentials to have in place

| What | Where | Needed |
|---|---|---|
| `xkmts.YYYYMMDD.NNNNNN_combined.csv` — four-chip DAOPHOT catalogues (`CHIP, ORIG_ID, X, Y, RA, DEC, MAG, MERR, MSKY, NITER, CHI, SHARP`) | working directory, matched by `frames_glob` | always |
| `xkmts.YYYYMMDD.NNNNNN.fits` — the frames, read for their headers only (airmass, date, object) | working directory | always, or instead a `frame_meta.csv` with columns `frame, airmass, date_obs, exptime, object` |
| `FilterTransmission.txt` — lab transmission of the filter (nm, then T% at nine positions) | working directory (`filter_file`) | always |
| `anet_api_key.txt` — astrometry.net API key (free account at nova.astrometry.net); or the environment variable `ASTROMETRY_NET_API_KEY` | working directory / shell | only for chips without a cached `<stem>_anet_wcs.hdr` |
| `flatB.fits` (master flat, extensions `K01`…`T08`), `alpha_lyr_stis_012.fits` (CALSPEC Vega) | working directory | optional (Sections 6 and 7) |

No other password or login is needed: the Gaia DR3 bulk files (ESA CDN), the Roman GBTDS layouts (GitHub) and the Gaia sky image
(CDS hips2fits) are public, and the Gaia archive queries used when `gspc_source = 'archive'` run anonymously. The first pass needs
internet access and tens of GB of disk for the Gaia and XP bulk files; everything downloaded is cached and reused.

### What the notebook does

For every frame of the program and every chip (K, M, N, T):

1. plate-solve the DAOPHOT catalogue (astrometry.net, then a TAN-SIP refinement against Gaia);
2. match the detections to Gaia DR3 (0.5", Gaia positions propagated to the epoch of the frame);
3. compute reference magnitudes $B_{\rm K}$ (AB) from the Gaia XP spectra of the matched stars through the KMTNet system passband
   (filter $\times$ measured chip QE);
4. fit the 15-coefficient model
   $\mathrm{MAG} - B_{\rm K} = a_0 + a_c\Delta c + a_{c^2}\Delta c^2 + a(x,y) + a_{\rm amp}$, with $\Delta c = (G_{\rm BP}-G_{\rm RP}) - 1.5$,
   five spatial terms and sum-to-zero amplifier offsets, by iteratively clipped least squares;
5. write a calibrated catalogue of every detection (with its Gaia match, $B_{\rm K}$ and a predicted Johnson $B$), a JSON record and a
   diagnostic figure.

Sections 4–6 summarise the program and compare repeated pointings and the master flat; Sections 7–9 produce the figures and
tables of the paper.

**Caches** (safe to delete, re-created on demand): `<stem>_anet_wcs.hdr`, `gaia_<field>_*.fits`, `gaia_bulk_files/`, `xp_spectra_<field>/`,
`xp_bulk_files/`, `gspc_*.csv`, `johnson_transformation.json`, `gbtds/`, `gaia_sky_background.npz`.
**Outputs**: `<stem>_calibrated_catalog.csv/.fits`, `<stem>_calibration.json`, `calibration_summary.csv`, `coefficients_table.tex`,
`figs/`.

**Running.** Edit the configuration below and run all cells. The batch is resumable: a chip-frame whose record was written by this
notebook version is only loaded, so re-running for the later sections costs seconds. Records from another version are recomputed
(5–25 s per chip-frame once the caches exist; from scratch, the astrometry.net solves and the downloads dominate).
Python $\geq$ 3.9 with `numpy, scipy, pandas, matplotlib, astropy, astroquery, astropy_healpix, gaiaxpy, requests`.

In [ ]:
# =============================================================================
# CONFIGURATION -- the only cell that normally needs editing
# =============================================================================
import os, io, re, glob, json, time, warnings, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.colors import LogNorm
from matplotlib.path import Path
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, PathPatch
from numpy.linalg import lstsq, inv
from scipy.spatial import cKDTree
from scipy.interpolate import RegularGridInterpolator
from astropy.table import Table, vstack
from astropy.coordinates import SkyCoord
from astropy.time import Time
from astropy.io import fits
from astropy.wcs import WCS
import astropy.units as u

CFG = dict(
    # ---- frames and chips ---------------------------------------------------------------------------------------------------------
    frames_glob='xkmts.2023071*_combined.csv',      # four-chip DAOPHOT catalogues; every match is one frame
    chips=('K', 'M', 'N', 'T'),
    max_frames=None,                                # process only the first N frames (for testing); None -> all
    skip_items=('xkmts.20230716.021046_combined_K', 'xkmts.20230716.021046_combined_T',
                'xkmts.20230716.021070_combined_T'),  # unmatchable detections; the repeats 021048 / 021072 cover these fields
    frame_meta_csv='frame_meta.csv',                # fallback for frames without a FITS file: columns frame, airmass, date_obs, exptime, object
    site='SAAO',                                    # KMTNet site of the frames (selects the measured chip QE)
    flat_file='flatB.fits',                         # optional master flat (Section 6)
    vega_file='alpha_lyr_stis_012.fits',            # optional CALSPEC Vega spectrum (Vega zero point, Section 7)
    image_width=9216, image_height=9232,            # chip size in pixels
    amp_width=1152,                                 # amplifier stripe width: 8 amplifiers per chip
    pixscale=0.3956,                                # nominal pixel scale [arcsec]
    merr_sentinel=9.0,                              # DAOPHOT marks failed measurements with MERR = 9.999
    # ---- photometric system -------------------------------------------------------------------------------------------------------
    filter_file='FilterTransmission.txt',
    xp_sampling=np.arange(336.0, 1021.0, 2.0),      # wavelength grid [nm] of the calibrated XP spectra
    # ---- astrometry.net, Gaia, matching -------------------------------------------------------------------------------------------
    anet_n_sources=1500, anet_mag_min=9.0, anet_scale=(0.38, 0.42), anet_search_radius_deg=5.0, anet_tweak_order=3,
    anet_timeout=600, anet_attempts=3,
    wcs_refine_search_arcmin=2.5, wcs_refine_sip=3, wcs_refine_grid=5,
    gaia_gmax=20.5,                                 # faint limit of the Gaia field catalogues
    gaia_box_half_deg=0.60,                         # half-width of the Gaia box around a chip (the chip is 1.0 deg wide)
    match_radius_arcsec=0.50,
    min_match_frac=0.6,                             # a chip matching fewer detections than this is re-refined, then flagged
    gspc_source='offline', gspc_timeout_s=180,      # GSPC Johnson B: 'offline' (GaiaXPy on the XP bulk files) or 'archive' (Gaia archive query)
    # ---- calibration sample and model ---------------------------------------------------------------------------------------------
    merr_max=0.02, chi_max=2.0, sharp_abs_max=1.0, mag_min=9.6,          # DAOPHOT quality cuts for calibration stars
    bk_err_max=0.03, cstar_abs_max=0.05, ruwe_max=1.4,                     # Gaia / XP quality cuts
    bp_range=(12.0, 17.5), bp_rp_range=(0.5, 4.0),                         # magnitude and colour range of the calibration stars
    bp_rp_ref=1.5,                                  # reference colour: the zero point a_0 refers to BP-RP = 1.5
    min_calib_stars=100,
    clip_sigma=3.0, clip_iters=5,
    spatial_terms=('y', 'y2', 'xy', 'x2y', 'xy2'),  # every term carries a factor y, so none is degenerate with the amplifier steps
    color_bins=np.arange(0.5, 4.01, 0.25),          # colour bins of the residual plots
    # ---- output -------------------------------------------------------------------------------------------------------------------
    show_frame_figures=True,                        # show the per-chip diagnostic figures inline (they are saved to figs/ either way)
    write_fits=True,                                # write the calibrated catalogues as FITS as well as csv
    fig_dpi=100,
    version=37,                                     # records written by another version are recomputed (37: single mean passband)
)

# ---- derived constants -------------------------------------------------------------------------------------------------------------
N_AMP = CFG['image_width'] // CFG['amp_width']                 # amplifiers per chip
X0c, Y0c = 0.5 * CFG['image_width'], 0.5 * CFG['image_height']  # chip centre, origin of the normalised coordinates
REF = CFG['bp_rp_ref']                                         # reference colour
K = 2                                                          # order of the colour polynomial
SP = CFG['spatial_terms']
NAMES = (['a'] + ['c{}'.format(k) for k in range(1, K + 1)] + ['s_' + t for t in SP]
         + ['g_{:02d}'.format(j + 1) for j in range(1, N_AMP)])   # coefficient names, in the order of the design matrix
GRID = CFG['xp_sampling']                                      # wavelength grid of the XP spectra and passbands [nm]
DLAM = np.gradient(GRID)                                       # its spacing [nm]

plt.rcParams.update({'figure.dpi': CFG['fig_dpi'], 'font.size': 14, 'axes.labelsize': 15, 'axes.titlesize': 15,
                     'legend.fontsize': 13, 'xtick.labelsize': 13, 'ytick.labelsize': 13})
from astropy.utils.exceptions import AstropyWarning
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=AstropyWarning)
os.makedirs('figs', exist_ok=True)

FRAME_FILES = sorted(glob.glob(CFG['frames_glob']))[:CFG['max_frames']]
print('{:d} frames x {:d} chips configured; model terms: {}'.format(len(FRAME_FILES), len(CFG['chips']), NAMES))

## 1. Detector QE and the system passband
The vendor's typical e2v CCD290-99 curve, digitised from the data sheet, is rescaled to pass through the measured QE of each chip at
350/400/500/650/900 nm; the system passband that defines $B_{\rm K}$ is the lab filter curve (mean of nine positions) times the mean
of the four chips' QE curves. (The four differ by less than 0.5 nm in effective wavelength, i.e. by less than 0.003 mag for the reddest
stars, so one passband serves all chips.)

In [ ]:
# ---- detector QE: the vendor's typical curve (digitised data sheet), rescaled to the measured QE of each chip -------------------------
QE_TYP_LAM = np.array([300, 320, 340, 350, 360, 370, 380, 390, 400, 410, 420, 450, 500, 550, 600, 650, 700, 730, 760, 800, 850, 900,
                       950, 1000, 1050, 1100], float)
QE_TYP = np.array([.20, .33, .47, .55, .62, .70, .80, .88, .91, .92, .925, .91, .90, .89, .89, .90, .92, .93, .93, .89, .80, .62, .40,
                   .13, .03, .0])
QE_MEAS_LAM = np.array([350, 400, 500, 650, 900], float)      # wavelengths [nm] of the measured QE values (%) below, per site and chip
QE_MEAS = {'SSO':  dict(K=[38.5, 83.3, 91.1, 90.9, 60.9], M=[40.1, 83.4, 90.1, 91.4, 59.8], T=[37.9, 83.6, 92.1, 91.5, 61.5], N=[44.3, 86.0, 89.6, 90.3, 60.7]),
           'CTIO': dict(K=[43.8, 87.8, 90.6, 94.2, 60.8], M=[49.3, 89.6, 90.8, 93.9, 59.9], T=[44.9, 89.3, 93.1, 93.1, 61.6], N=[42.9, 88.1, 93.5, 94.2, 64.2]),
           'SAAO': dict(K=[41.0, 86.1, 93.6, 93.6, 65.6], M=[44.6, 89.4, 95.2, 97.1, 63.6], T=[43.3, 85.2, 89.5, 90.1, 58.7], N=[58.9, 90.9, 93.5, 97.0, 62.0])}


def qe_curve(lam, site, chip):
    '''QE of one chip at wavelengths lam [nm]: the typical curve times the ratio measured/typical, interpolated in wavelength
    between the five measured points and held constant outside 350-900 nm.'''
    ratio = np.array(QE_MEAS[site][chip]) / 100.0 / np.interp(QE_MEAS_LAM, QE_TYP_LAM, QE_TYP)
    return np.interp(lam, QE_TYP_LAM, QE_TYP) * np.interp(lam, QE_MEAS_LAM, ratio)


# ---- the system passband: filter transmission (mean of the nine lab positions) times the mean QE of the four chips, on the XP grid --------
# The four chips' QE curves differ by a slight tilt that would move the effective wavelength by < 0.5 nm and the magnitude of the
# reddest stars by < 0.003 mag, so one passband -- the mean -- defines B_K for all chips.
raw = np.loadtxt(CFG['filter_file'], comments='#')
S_FILTER = np.interp(GRID, raw[:, 0], (raw[:, 1:] / 100.0).mean(axis=1), left=0.0, right=0.0)
QE_MEAN = np.mean([qe_curve(GRID, CFG['site'], ch) for ch in CFG['chips']], axis=0)
S_SYS = S_FILTER * QE_MEAN
LAM_EFF = float(np.sum(S_SYS * GRID ** 2 * DLAM) / np.sum(S_SYS * GRID * DLAM))      # photon-weighted effective wavelength
PASSBAND = {ch: dict(S=S_SYS, lam_eff=LAM_EFF) for ch in CFG['chips']}             # one system; keyed by chip for convenience
half_max = GRID[S_SYS > 0.5 * S_SYS.max()]
print('system passband: lambda_eff {:.1f} nm, half-maximum {:.0f}-{:.0f} nm'.format(LAM_EFF, half_max[0], half_max[-1]))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
lam_p = np.arange(300, 1101, 2.0)
for ch in CFG['chips']:
    ax[0].plot(lam_p, qe_curve(lam_p, CFG['site'], ch), lw=0.8, alpha=0.7, label='{}-{}'.format(CFG['site'], ch))
ax[0].plot(lam_p, np.mean([qe_curve(lam_p, CFG['site'], ch) for ch in CFG['chips']], axis=0), 'k-', lw=1.8, label='mean (adopted)')
ax[0].plot(QE_TYP_LAM, QE_TYP, 'k--', lw=0.8, label='typical (vendor)')
ax[0].set(xlabel='wavelength [nm]', ylabel='QE', title='detector QE')
ax[0].legend()
ax[1].plot(GRID, S_FILTER, 'k-', label='filter')
ax[1].plot(GRID, QE_MEAN, 'r--', label='mean QE')
ax[1].plot(GRID, S_SYS, 'b-', lw=1.8, label='system passband ({:.0f} nm)'.format(LAM_EFF))
ax[1].set(xlim=(350, 560), xlabel='wavelength [nm]', ylabel='response', title='system passband')
ax[1].legend(fontsize=10)
plt.tight_layout()
plt.show()

## 2. Pipeline functions
One function per step; `run_frame()` chains them. Nothing here needs editing for normal use.

In [ ]:
# =============================================================================
# 2a. Catalogue and frame metadata
# =============================================================================

def frame_id(csv):
    '''Frame identifier, e.g. 'xkmts.20230715.020746', from a catalogue file name.'''
    return re.search(r'(xkmt[a-z]\.\d{8}\.\d{6})', os.path.basename(csv)).group(1)


def stem_of(csv, chip):
    '''Stem of all files belonging to one (frame, chip), e.g. 'xkmts.20230715.020746_combined_T'.'''
    return '{}_{}'.format(os.path.splitext(os.path.basename(csv))[0], chip)


def frame_meta(fid, chip):
    '''Airmass, date, mid-exposure epoch, exposure time and object of a frame: from the FITS header (primary header updated with
    the first extension of the chip) if the frame is present, else from the row of frame_meta_csv.'''
    f = fid + '.fits'
    if os.path.exists(f):
        with fits.open(f) as hdul:
            h = fits.Header(hdul[0].header)
            ext = next((x for x in hdul[1:] if x.name.upper().startswith(chip)), None)
            if ext is not None:
                h.update(ext.header)
        airmass = float(h.get('SECZ', h.get('AIRMASS', np.nan)))
        date_obs = str(h['DATE-OBS'])
        exptime = float(h.get('EXPTIME', 0))
        obj = h.get('OBJECT')
    else:
        t = pd.read_csv(CFG['frame_meta_csv'], dtype={'frame': str}).set_index('frame')
        if fid not in t.index:
            raise RuntimeError('{}: no FITS file and no row in {}'.format(fid, CFG['frame_meta_csv']))
        r = t.loc[fid]
        airmass = float(r['airmass'])
        date_obs = str(r['date_obs'])
        exptime = float(r.get('exptime', 0))
        obj = r.get('object')
    epoch = float((Time(date_obs, scale='utc') + exptime / 2 * u.s).decimalyear)
    return dict(frame=fid, chip=chip, airmass=airmass, date_obs=date_obs[:10], epoch=epoch, exptime=exptime,
                object=None if obj is None else str(obj))


def load_frame(csv):
    '''A four-chip DAOPHOT csv as a DataFrame: sentinel rows (MERR = 9.999) removed, ORIG_ID renamed to ID.'''
    cat = pd.read_csv(csv)
    return cat.loc[cat['MERR'] < CFG['merr_sentinel']].rename(columns={'ORIG_ID': 'ID'}).reset_index(drop=True)


def chip_catalogue(cat, chip):
    '''The rows of one chip, with the amplifier index of every star. The pipeline RA/DEC are kept only as a starting guess for the
    astrometry (RA_orig, DEC_orig); the columns RA, DEC are set later from the WCS.'''
    df = cat.loc[cat['CHIP'].astype(str).str.upper() == chip].drop(columns='CHIP')
    df = df.rename(columns={'RA': 'RA_orig', 'DEC': 'DEC_orig'}).reset_index(drop=True)
    if len(df) == 0:
        raise RuntimeError('no rows for chip ' + chip)
    df['amp'] = np.clip((df['X'] // CFG['amp_width']).astype(int), 0, N_AMP - 1)
    return df

In [ ]:
# =============================================================================
# 2b. Astrometry: astrometry.net solution, refined against Gaia
# =============================================================================

def anet_key():
    '''The astrometry.net API key: the environment variable, else the content of anet_api_key.txt, else None.'''
    k = os.environ.get('ASTROMETRY_NET_API_KEY')
    if k:
        return k
    return open('anet_api_key.txt').read().strip() if os.path.exists('anet_api_key.txt') else None


def solve_astrometry_net(df, stem):
    '''Blind solution from astrometry.net (nova): the brightest well-measured stars of every ninth of the chip are submitted, so
    that the solution is constrained over the whole chip. Returns a FITS header with the TAN-SIP solution.'''
    if not anet_key():
        raise RuntimeError('no astrometry.net API key (anet_api_key.txt or $ASTROMETRY_NET_API_KEY)')
    from astroquery.astrometry_net import AstrometryNet
    ast = AstrometryNet()
    ast.api_key = anet_key()
    # source list: good DAOPHOT measurements, the brightest n/9 in each of the 3 x 3 chip cells
    src = df[(df['MAG'] > CFG['anet_mag_min']) & (df['CHI'] < CFG['chi_max']) & (np.abs(df['SHARP']) < CFG['sharp_abs_max'])]
    cell = (np.clip(src['X'] // (CFG['image_width'] / 3), 0, 2) * 3 + np.clip(src['Y'] // (CFG['image_height'] / 3), 0, 2)).astype(int)
    src = src.assign(cell=cell).sort_values('MAG').groupby('cell').head(int(np.ceil(CFG['anet_n_sources'] / 9)))
    settings = dict(scale_units='arcsecperpix', scale_type='ul', scale_lower=CFG['anet_scale'][0], scale_upper=CFG['anet_scale'][1],
                    center_ra=float(np.median(df['RA_orig'])), center_dec=float(np.median(df['DEC_orig'])),
                    radius=CFG['anet_search_radius_deg'], tweak_order=CFG['anet_tweak_order'], crpix_center=True, parity=2,
                    publicly_visible='n')
    print('    astrometry.net: solving {} ({:d} sources) ...'.format(stem, len(src)), flush=True)
    for attempt in range(1, CFG['anet_attempts'] + 1):                       # nova drops connections now and then
        try:
            try:
                hdr = ast.solve_from_source_list(src['X'].values, src['Y'].values, CFG['image_width'], CFG['image_height'],
                                                 solve_timeout=CFG['anet_timeout'], **settings)
            except TimeoutError as e:                                         # the job is still running: keep waiting for it
                hdr = ast.monitor_submission(e.args[1], solve_timeout=CFG['anet_timeout'])
            if hdr:
                return hdr
            raise RuntimeError('no solution returned')
        except Exception as e:
            if attempt == CFG['anet_attempts']:
                raise RuntimeError('astrometry.net failed for {}: {}'.format(stem, str(e)[:80]))
            print('    attempt {} failed ({}); retrying in 120 s'.format(attempt, str(e)[:60]), flush=True)
            time.sleep(120)


def refine_wcs_with_gaia(df, wcs0, gdf, label):
    '''Refine a starting WCS against Gaia (G < 17).

    1. The shift between the starting frame and the sky is measured by cross-correlating the bright stars with the projected Gaia
       positions in each cell of a grid over the chip, and interpolated to every star -- so a start that is off by tens of pixels
       in one corner still matches everywhere.
    2. A TAN-SIP WCS is fitted to the matches, with a matching radius shrinking from 15 to 1.5 pixels.
    Returns (header, info) or (None, info) when the refinement does not converge.'''
    from astropy.wcs.utils import fit_wcs_from_points
    W, H, NG, px = CFG['image_width'], CFG['image_height'], CFG['wcs_refine_grid'], CFG['pixscale']
    # Gaia stars that fall on (or within 400 px of) the chip in the starting frame
    gb = gdf[gdf['phot_g_mean_mag'] < 17.0]
    gra, gde = gb['ra'].values, gb['dec'].values
    gx, gy = wcs0.all_world2pix(gra, gde, 1, quiet=True)
    ok = np.isfinite(gx) & np.isfinite(gy) & (gx > -400) & (gx < W + 400) & (gy > -400) & (gy < H + 400)
    gra, gde, gx, gy = gra[ok], gde[ok], gx[ok], gy[ok]
    if len(gra) < 300:
        return None, dict(reason='only {:d} Gaia stars project onto the chip'.format(len(gra)))
    X, Y, mags = df['X'].values, df['Y'].values, df['MAG'].values
    bright = df.nsmallest(min(8000, len(df)), 'MAG')
    bx, by = bright['X'].values, bright['Y'].values
    # step 1: shift per grid cell from the peak of the (dx, dy) histogram of all star-Gaia pairs within the search window
    gtree = cKDTree(np.c_[gx, gy])
    win = CFG['wcs_refine_search_arcmin'] * 60.0 / px
    xc, yc = (np.arange(NG) + 0.5) * W / NG, (np.arange(NG) + 0.5) * H / NG
    SX, SY = np.full((NG, NG), np.nan), np.full((NG, NG), np.nan)
    for i in range(NG):
        for j in range(NG):
            cell = (bx >= i * W / NG) & (bx < (i + 1) * W / NG) & (by >= j * H / NG) & (by < (j + 1) * H / NG)
            pairs = [(gx[p] - bx[cell][k], gy[p] - by[cell][k])
                     for k, p in enumerate(gtree.query_ball_point(np.c_[bx[cell], by[cell]], r=win)) if len(p)]
            if not pairs:
                continue
            dx, dy = np.concatenate([a for a, _ in pairs]), np.concatenate([b for _, b in pairs])
            Hh, xe, ye = np.histogram2d(dx, dy, bins=int(win / 2) + 1, range=[[-win, win], [-win, win]])
            a, b = np.unravel_index(np.argmax(Hh), Hh.shape)
            f = (np.abs(dx - 0.5 * (xe[a] + xe[a + 1])) < 6) & (np.abs(dy - 0.5 * (ye[b] + ye[b + 1])) < 6)
            if f.sum() >= 15:
                SX[i, j], SY[i, j] = np.median(dx[f]), np.median(dy[f])
    n_ok = int(np.isfinite(SX).sum())
    if n_ok < 4:
        return None, dict(reason='cross-correlation peak found in only {:d} cells'.format(n_ok))
    for S in (SX, SY):                                                        # empty cells take the nearest measured shift
        ii, jj = np.where(np.isfinite(S))
        for a, b in zip(*np.where(~np.isfinite(S))):
            k = np.argmin((ii - a) ** 2 + (jj - b) ** 2)
            S[a, b] = S[ii[k], jj[k]]
    sx = RegularGridInterpolator((xc, yc), SX, bounds_error=False, fill_value=None)(np.c_[X, Y])
    sy = RegularGridInterpolator((xc, yc), SY, bounds_error=False, fill_value=None)(np.c_[X, Y])
    info = dict(shift_px=(float(np.nanmedian(SX)), float(np.nanmedian(SY))))
    # step 2: iterative TAN-SIP fit to the nearest-neighbour matches, brightest 10000 stars
    wcs = None
    for radius in (15.0, 6.0, 3.0, 1.5, 1.5, 1.5):
        if wcs is None:                                                       # first pass: shifted starting frame
            gxs, gys, qx, qy = gx, gy, X + sx, Y + sy
        else:                                                                 # later passes: the current fit
            gxs, gys = wcs.all_world2pix(gra, gde, 1, quiet=True)
            qx, qy = X, Y
        d, idx = cKDTree(np.c_[gxs, gys]).query(np.c_[qx, qy], distance_upper_bound=radius)
        m = np.isfinite(d)
        if m.sum() < 100:
            return None, dict(reason='only {:d} matches at {:.0f} px'.format(int(m.sum()), radius), **info)
        sel = np.where(m)[0]
        sel = sel[np.argsort(mags[sel])][:10000]
        wcs = fit_wcs_from_points((X[sel] - 1, Y[sel] - 1), SkyCoord(gra[idx[sel]] * u.deg, gde[idx[sel]] * u.deg),
                                  sip_degree=CFG['wcs_refine_sip'])
    # quality: fraction of the bright stars with a Gaia star within 1.5" in the final solution, and their rms distance
    gxs, gys = wcs.all_world2pix(gra, gde, 1, quiet=True)
    d, _ = cKDTree(np.c_[gxs, gys]).query(np.c_[bx, by], distance_upper_bound=1.5 / px)
    okm = np.isfinite(d)
    info.update(n_fit=int(len(sel)), bright_match_frac=float(okm.mean()),
                rms_arcsec=float(np.sqrt(np.mean(d[okm] ** 2)) * px) if okm.any() else np.nan)
    hdr = wcs.to_header(relax=True)
    hdr['WCSSRC'] = ('gaia-refined ' + label)[:60]
    return hdr, info


def refined_ok(hdr, info):
    '''Acceptance criterion for a refined solution: > 90% of the bright stars matched, rms < 0.25".'''
    return hdr is not None and info['bright_match_frac'] > 0.9 and info['rms_arcsec'] < 0.25


def good_record(stem):
    '''True if stem has a record with a trustworthy astrometric solution.'''
    jf = stem + '_calibration.json'
    if not (os.path.exists(jf) and os.path.exists(stem + '_anet_wcs.hdr')):
        return False
    r = json.load(open(jf))
    return (not r.get('astrometry_suspect')) and r.get('match', {}).get('match_frac', 0) > CFG['min_match_frac']


def sibling_header(obj, chip, stem):
    '''Cached WCS of another frame of the same field and chip (the same pointing to a few pixels), if any.'''
    for jf in glob.glob('*_{}_calibration.json'.format(chip)):
        r = json.load(open(jf))
        if obj and r.get('object') == obj and r['stem'] != stem and good_record(r['stem']):
            return r['stem'] + '_anet_wcs.hdr'


def apply_wcs(df, hdr):
    '''Set RA, DEC of every star from the header; return the astrometry summary (pixel scale, SIP order, centre, source).'''
    w = WCS(hdr)
    df['RA'], df['DEC'] = w.all_pix2world(df['X'].values, df['Y'].values, 1)
    cd = w.pixel_scale_matrix * 3600.0
    return dict(pixscale=float(np.hypot(cd[0, 0], cd[1, 0])), sip=int(hdr.get('A_ORDER', 0)),
                ra_c=float(w.wcs.crval[0]), dec_c=float(w.wcs.crval[1]), source=str(hdr.get('WCSSRC', 'astrometry.net')))


def solve_wcs(df, meta):
    '''Astrometric solution of one chip, in order of preference: the cached header; the Gaia-refined solution of a sibling frame;
    the astrometry.net solution polished against Gaia (kept unpolished if the refinement does not converge -- the match fraction
    then flags the record). The accepted header is cached as <stem>_anet_wcs.hdr.'''
    stem = meta['stem']
    hf = stem + '_anet_wcs.hdr'
    if not os.path.exists(hf):
        sib = sibling_header(meta['object'], meta['chip'], stem)
        starts = [(lambda: fits.Header.fromtextfile(sib), 'sibling ' + sib[:28])] if sib else []
        starts.append((lambda: solve_astrometry_net(df, stem), 'astrometry.net'))
        raw = None
        for get_header, label in starts:
            hdr0 = get_header()
            apply_wcs(df, hdr0)
            gdf, _ = gaia_field(df)
            hdr, info = refine_wcs_with_gaia(df, WCS(hdr0), gdf, label)
            if refined_ok(hdr, info):
                print('    {}: {} solution refined against Gaia ({:d} stars, rms {:.3f}", bright-star match {:.0%})'.format(
                      stem, label, info['n_fit'], info['rms_arcsec'], info['bright_match_frac']), flush=True)
                hdr.totextfile(hf, overwrite=True)
                break
            why = info.get('reason') or 'rms {:.3f}", match {:.0%}'.format(info['rms_arcsec'], info['bright_match_frac'])
            print('    {}: refinement of the {} solution rejected ({})'.format(stem, label, why), flush=True)
            if label == 'astrometry.net':
                raw = hdr0
        else:                                                                 # no refinement accepted
            if raw is None:
                raise RuntimeError('no astrometric solution for ' + stem)
            raw.totextfile(hf, overwrite=True)
            print('    {}: keeping the unrefined astrometry.net solution'.format(stem), flush=True)
    return apply_wcs(df, fits.Header.fromtextfile(hf))

In [ ]:
# =============================================================================
# 2c. Gaia DR3: field catalogues from the ESA bulk files, matching, GSPC
# =============================================================================
GS_BULK = 'https://cdn.gea.esac.esa.int/Gaia/gdr3/gaia_source/'
GS_LIST = 'https://gaia.eu-1.cdn77-storage.com/?prefix=Gaia/gdr3/gaia_source/&delimiter=/'
GS_COLS = ['source_id', 'ra', 'dec', 'l', 'b', 'parallax', 'parallax_error', 'parallax_over_error', 'pmra', 'pmra_error', 'pmdec',
           'pmdec_error', 'astrometric_params_solved', 'ruwe', 'ipd_frac_multi_peak', 'phot_g_mean_mag', 'phot_bp_mean_mag',
           'phot_rp_mean_mag', 'bp_rp', 'phot_bp_rp_excess_factor', 'phot_variable_flag', 'has_xp_continuous']
GAIA_COLS = GS_COLS + ['c_star']                                             # Gaia columns carried into the calibrated catalogues
STRING_COLS = ('phot_variable_flag',)


def cdn_listing(url):
    '''All keys of a (paginated) ESA CDN bucket listing.'''
    keys, marker = [], ''
    while True:
        txt = requests.get(url + ('&marker=' + marker if marker else ''), timeout=120).text
        page = re.findall(r'<Key>(.*?)</Key>', txt)
        keys += page
        if '<IsTruncated>true</IsTruncated>' not in txt or not page:
            return keys
        marker = page[-1]


def download_resumable(url, local, attempts=5):
    '''Stream a file to local, resuming a partial download with a Range request if the transfer breaks.'''
    for attempt in range(1, attempts + 1):
        try:
            have = os.path.getsize(local + '.part') if os.path.exists(local + '.part') else 0
            headers = {'Range': 'bytes={:d}-'.format(have)} if have else {}
            with requests.get(url, stream=True, timeout=120, headers=headers) as r:
                if r.status_code == 416:                                      # the partial file is already complete
                    break
                r.raise_for_status()
                with open(local + '.part', 'ab' if (have and r.status_code == 206) else 'wb') as fh:
                    for blk in r.iter_content(chunk_size=1 << 20):
                        fh.write(blk)
            break
        except Exception as e:
            if attempt == attempts:
                raise
            print('    download of {} interrupted ({}); attempt {}/{}'.format(os.path.basename(local), str(e)[:50], attempt, attempts), flush=True)
            time.sleep(20)
    os.replace(local + '.part', local)


def fits_safe(t):
    '''Object/bytes string columns of an astropy Table -> fixed-width unicode, so that the table can be written to FITS.'''
    for c in t.colnames:
        if t[c].dtype.kind in 'OSU':
            vals = ['' if v is None or v is np.ma.masked else (v.decode() if isinstance(v, bytes) else str(v)) for v in t[c]]
            t[c] = np.array(vals, dtype='U{:d}'.format(max(1, max(len(v) for v in vals))))
    return t


def field_box(df):
    '''Field key and Gaia box (deg) covering the chip. The box centre is rounded to 0.05 deg so that repeated pointings of the same
    field share one cached catalogue.'''
    hw = CFG['gaia_box_half_deg']
    ra_c, dec_c = [0.05 * round(0.5 * (df[c].min() + df[c].max()) / 0.05) for c in ('RA', 'DEC')]
    box = dict(ra_lo=ra_c - hw / np.cos(np.deg2rad(dec_c)), ra_hi=ra_c + hw / np.cos(np.deg2rad(dec_c)), de_lo=dec_c - hw, de_hi=dec_c + hw)
    inside = (box['ra_lo'] < df['RA'].min() and df['RA'].max() < box['ra_hi'] and box['de_lo'] < df['DEC'].min() and df['DEC'].max() < box['de_hi'])
    if not inside:
        raise RuntimeError('chip footprint exceeds the Gaia box; increase gaia_box_half_deg')
    return 'RA{:.2f}_Dec{:+.2f}'.format(ra_c, dec_c), box


def gaia_from_bulk(box, cache):
    '''Field catalogue from the GaiaSource bulk files covering the box (the files are named by their HEALPix level-8 ranges).
    Each bulk file is reduced once to a light FITS (needed columns, G < 21) in gaia_bulk_files/ and reused by every field it covers.'''
    from astropy_healpix import HEALPix
    ra_g, de_g = np.meshgrid(np.linspace(box['ra_lo'], box['ra_hi'], 60), np.linspace(box['de_lo'], box['de_hi'], 60))
    pix = np.unique(HEALPix(nside=256, order='nested').lonlat_to_healpix(ra_g.ravel() * u.deg, de_g.ravel() * u.deg))
    files = sorted({(f, int(lo), int(hi)) for f, lo, hi in re.findall(r'(GaiaSource_(\d+)-(\d+)\.csv\.gz)', ' '.join(cdn_listing(GS_LIST)))})
    files = [f for f, lo, hi in files if np.any((pix >= lo) & (pix <= hi))]
    os.makedirs('gaia_bulk_files', exist_ok=True)
    print('    Gaia bulk: {:d} GaiaSource files cover the field'.format(len(files)), flush=True)
    parts = []
    for fname in files:
        lite = os.path.join('gaia_bulk_files', fname.replace('.csv.gz', '_lite.fits'))
        if not os.path.exists(lite):
            gz = os.path.join('gaia_bulk_files', fname)
            if not os.path.exists(gz):
                t0 = time.time()
                download_resumable(GS_BULK + fname, gz)
                print('    downloaded {} ({:.0f} s)'.format(fname, time.time() - t0), flush=True)
            chunks = pd.read_csv(gz, compression='gzip', comment='#', usecols=GS_COLS, chunksize=200000)
            d = pd.concat([c[c['phot_g_mean_mag'] < 21.0] for c in chunks], ignore_index=True)
            d['has_xp_continuous'] = d['has_xp_continuous'].astype(str)
            fits_safe(Table.from_pandas(d)).write(lite, overwrite=True)
            os.remove(gz)
        t = Table.read(lite)
        sel = ((t['ra'] >= box['ra_lo']) & (t['ra'] < box['ra_hi']) & (t['dec'] >= box['de_lo']) & (t['dec'] < box['de_hi'])
               & (t['phot_g_mean_mag'] < CFG['gaia_gmax']))
        sel = np.asarray(sel, bool)
        if sel.any():
            parts.append(t[sel])
    t = vstack(parts, metadata_conflicts='silent')
    t = fits_safe(t[np.sort(np.unique(np.asarray(t['source_id']), return_index=True)[1])])     # one row per source
    t.write(cache, overwrite=True)
    print('    Gaia bulk: {:d} sources with G < {} cached to {}'.format(len(t), CFG['gaia_gmax'], cache), flush=True)
    return t


def c_star(excess, bp_rp):
    '''Corrected BP/RP flux-excess factor C* of Riello et al. (2021, eq. 6): the excess factor minus its colour trend for
    well-behaved sources; |C*| > 0.05 flags blended or non-stellar photometry.'''
    x = np.asarray(bp_rp, float)
    f = np.where(x < 0.5, 1.154360 + 0.033772 * x + 0.032277 * x ** 2,
                 np.where(x < 4.0, 1.162004 + 0.011464 * x + 0.049255 * x ** 2 - 0.005879 * x ** 3, 1.057572 + 0.140537 * x))
    return np.asarray(excess, float) - f


def gaia_field(df):
    '''Cached Gaia DR3 catalogue of the field as a DataFrame (G < gaia_gmax), plus the field key. Caches built by the earlier archive
    route (GSPC columns, no has_xp_continuous) are accepted: a GSPC row exists exactly when an XP spectrum was published.'''
    key, box = field_box(df)
    hw = CFG['gaia_box_half_deg']
    cands = sorted(f for f in glob.glob('gaia_{}_r{:.2f}_G*.fits'.format(key, hw))
                   if float(re.search(r'_G([\d.]+)\.fits$', f).group(1)) >= CFG['gaia_gmax'])    # an existing deeper cache serves too
    t = Table.read(cands[0]) if cands else gaia_from_bulk(box, 'gaia_{}_r{:.2f}_G{:.1f}.fits'.format(key, hw, CFG['gaia_gmax']))
    g = t.to_pandas()
    g.columns = [c.lower() for c in g.columns]
    g = g[g['phot_g_mean_mag'] < CFG['gaia_gmax']].reset_index(drop=True)
    for c in g.columns:
        if g[c].dtype == object:
            g[c] = g[c].map(lambda v: v.decode() if isinstance(v, bytes) else v)
    if 'c_star' not in g.columns:
        g['c_star'] = c_star(g['phot_bp_rp_excess_factor'], g['bp_rp'])
    if 'has_xp_continuous' in g.columns:
        g['has_xp_continuous'] = g['has_xp_continuous'].astype(str).str.strip().str.lower().isin(('true', '1', 't'))
    else:
        g['has_xp_continuous'] = g['c_star'].notna()
    for c in GS_COLS:
        if c not in g.columns:
            g[c] = np.nan
    return g, key


def match_gaia(df, gdf, epoch):
    '''Nearest Gaia source within match_radius of every star (Gaia positions propagated to the epoch), one star per Gaia source
    (the closest). Returns the matched table and the match statistics.'''
    dt = epoch - 2016.0
    dec_g = gdf['dec'].values + np.nan_to_num(gdf['pmdec'].values) * dt / 3.6e6
    ra_g = gdf['ra'].values + np.nan_to_num(gdf['pmra'].values) * dt / 3.6e6 / np.cos(np.deg2rad(dec_g))
    idx, sep2d, _ = SkyCoord(df['RA'].values * u.deg, df['DEC'].values * u.deg).match_to_catalog_sky(SkyCoord(ra_g * u.deg, dec_g * u.deg))
    within = sep2d.arcsec < CFG['match_radius_arcsec']
    m = df.loc[within].assign(df_index=df.index[within], sep_arcsec=sep2d.arcsec[within]).reset_index(drop=True)
    m = m.join(gdf.iloc[idx[within]].reset_index(drop=True))
    m = m.sort_values('sep_arcsec').drop_duplicates('source_id').sort_values('df_index').reset_index(drop=True)
    dra = (m['ra'] - m['RA']) * np.cos(np.deg2rad(m['DEC'])) * 3600.0
    ddec = (m['dec'] - m['DEC']) * 3600.0
    mad = lambda v: 1.4826 * np.median(np.abs(v - np.median(v)))
    stats = dict(n_kmt=len(df), n_matched=len(m), match_frac=len(m) / len(df), sep_median=float(np.median(m['sep_arcsec'])),
                 astrom_sigma=float(np.hypot(mad(dra), mad(ddec)) / np.sqrt(2)))
    return m, stats


def gspc_archive(ids):
    '''GSPC rows of the given source_ids from the Gaia archive: synchronous upload queries of 2000 sources (no job queue, each
    abandoned after gspc_timeout_s), falling back to an asynchronous job for a chunk that fails.'''
    from astroquery.gaia import Gaia
    from concurrent.futures import ThreadPoolExecutor, TimeoutError as Timeout
    Gaia.ROW_LIMIT = -1
    q = ('SELECT sp.source_id, sp.b_jkc_mag, sp.b_jkc_flux, sp.b_jkc_flux_error, sp.b_jkc_flag, sp.v_jkc_mag '
         'FROM tap_upload.ids AS u JOIN gaiadr3.synthetic_photometry_gspc AS sp ON sp.source_id = u.source_id')
    out = []
    for k in range(0, len(ids), 2000):
        up = Table({'source_id': ids[k:k + 2000]})
        res = None
        for launch in (Gaia.launch_job, Gaia.launch_job_async):
            ex = ThreadPoolExecutor(max_workers=1)
            fut = ex.submit(lambda: launch(q, upload_resource=up, upload_table_name='ids').get_results().to_pandas())
            try:
                res = fut.result(timeout=CFG['gspc_timeout_s'] if launch is Gaia.launch_job else None)
                ex.shutdown(wait=False)
                break
            except (Timeout, Exception) as e:
                ex.shutdown(wait=False)
                print('    GSPC {}: {}'.format(launch.__name__, str(e)[:70] or 'no answer within {:d} s'.format(CFG['gspc_timeout_s'])), flush=True)
        if res is None:
            raise RuntimeError('Gaia archive unavailable (set gspc_source to offline to use the XP bulk files on disk instead)')
        out.append(res)
        print('    GSPC: {:d}/{:d} sources from the archive'.format(min(k + 2000, len(ids)), len(ids)), flush=True)
    g = pd.concat(out, ignore_index=True)
    g.columns = [c.lower() for c in g.columns]
    return g


def gspc_offline(ids):
    '''Standardised JKC B and V generated with GaiaXPy (PhotometricSystem.JKC_Std, the recipe behind GSPC) from the XP bulk files
    (those on disk, downloaded if missing); the GSPC validation flag is not available this way and is set to 1.'''
    from gaiaxpy import generate, PhotometricSystem
    print('    GSPC offline: extracting the XP spectra of {:d} sources from the bulk files (minutes per file) ...'.format(len(ids)), flush=True)
    tmp = xp_rows(ids)
    if tmp is None:
        raise RuntimeError('none of these sources is in the XP bulk files')
    ph = generate(tmp, photometric_system=PhotometricSystem.JKC_Std, save_file=False)
    os.remove(tmp)
    col = lambda kind, band: next(c for c in ph.columns if c.endswith('_{}_{}'.format(kind, band)))
    return pd.DataFrame(dict(source_id=ph['source_id'].astype(np.int64), b_jkc_mag=ph[col('mag', 'B')], b_jkc_flux=ph[col('flux', 'B')],
                             b_jkc_flux_error=ph[col('flux_error', 'B')], b_jkc_flag=1, v_jkc_mag=ph[col('mag', 'V')]))


def fetch_gspc(source_ids, cache):
    '''GSPC synthetic Johnson B and V of the given sources (archive or offline, see CFG gspc_source), cached to csv.'''
    if os.path.exists(cache):
        return pd.read_csv(cache)
    ids = np.unique(np.asarray(source_ids, dtype=np.int64))
    g = gspc_offline(ids) if CFG['gspc_source'] == 'offline' else gspc_archive(ids)
    g['b_jkc_mag_err'] = 1.0857362 * g['b_jkc_flux_error'] / g['b_jkc_flux']
    g.to_csv(cache, index=False)
    return g


def with_gspc(df, cache):
    '''df joined with the GSPC magnitudes of its source_ids (GSPC columns already present, e.g. from an archive-built cache, are replaced).'''
    df = df.drop(columns=[c for c in df.columns if c.startswith(('b_jkc', 'v_jkc'))])
    return df.merge(fetch_gspc(df['source_id'].values, cache), on='source_id')

In [ ]:
# =============================================================================
# 2d. XP spectra, reference magnitudes B_K, Johnson transformation
# =============================================================================
XP_BULK = 'https://cdn.gea.esac.esa.int/Gaia/gdr3/Spectroscopy/xp_continuous_mean_spectrum/'
XP_LIST = 'https://gaia.eu-1.cdn77-storage.com/?prefix=Gaia/gdr3/Spectroscopy/xp_continuous_mean_spectrum/&delimiter=/'


def load_xp_cache(d):
    '''All cached spectra of a field directory: (source_id, flux, flux_error) with the spectra as rows on GRID.'''
    z = [np.load(f) for f in sorted(glob.glob(os.path.join(d, '*.npz')))]
    if not z:
        return np.array([], dtype=np.int64), np.zeros((0, len(GRID)), np.float32), np.zeros((0, len(GRID)), np.float32)
    return (np.concatenate([x['source_id'] for x in z]), np.vstack([x['flux'] for x in z]), np.vstack([x['flux_error'] for x in z]))


def xp_bulk_covering(hp8):
    '''Local paths of the XP bulk files covering the given HEALPix-8 pixels: the ones already in xp_bulk_files/, the others downloaded
    (the files are named by their HEALPix-8 range, so the CDN listing is only consulted when a file is missing).'''
    rng = lambda f: tuple(int(v) for v in re.search(r'_(\d+)-(\d+)\.csv\.gz$', f).groups())
    local = {f for f in glob.glob('xp_bulk_files/XpContinuousMeanSpectrum_*.csv.gz') if np.any((hp8 >= rng(f)[0]) & (hp8 <= rng(f)[1]))}
    covered = np.zeros(len(hp8), bool)
    for f in local:
        covered |= (hp8 >= rng(f)[0]) & (hp8 <= rng(f)[1])
    if not covered.all():
        for fname in sorted(set(re.findall(r'XpContinuousMeanSpectrum_\d+-\d+\.csv\.gz', ' '.join(cdn_listing(XP_LIST))))):
            lo, hi = rng(fname)
            f = os.path.join('xp_bulk_files', fname)
            if np.any((hp8[~covered] >= lo) & (hp8[~covered] <= hi)) and not os.path.exists(f):
                t0 = time.time()
                download_resumable(XP_BULK + fname, f)
                print('    downloaded {} ({:.0f} s)'.format(fname, time.time() - t0), flush=True)
                local.add(f)
    return sorted(local)


def xp_rows(source_ids):
    '''The XP continuous-spectrum rows (bulk-file format) of the given sources, written to a temporary csv for GaiaXPy; returns its
    path, or None if none of the sources is in the bulk files.'''
    ids = np.unique(np.asarray(source_ids, np.int64))
    want = set(int(v) for v in ids)
    kept = []
    for f in xp_bulk_covering(np.unique(ids >> 43)):                          # source_id >> 43 is the HEALPix level-8 pixel
        for attempt in range(2):                                              # a truncated gzip is deleted and fetched again once
            try:
                kept += [p[p['source_id'].isin(want)] for p in pd.read_csv(f, compression='gzip', comment='#', chunksize=20000)]
                break
            except (EOFError, OSError, pd.errors.ParserError, ValueError) as e:
                print('    {} unreadable ({}); re-downloading'.format(os.path.basename(f), str(e)[:60]), flush=True)
                os.remove(f)
                if attempt == 1:
                    raise
                download_resumable(XP_BULK + os.path.basename(f), f)
    kept = [k for k in kept if len(k)]
    if not kept:
        return None
    tmp = 'xp_bulk_files/subset_{:d}.csv'.format(int(time.time()))
    pd.concat(kept, ignore_index=True).to_csv(tmp, index=False)
    return tmp


def xp_spectra(source_ids, field_key):
    '''Calibrated XP spectra (W m^-2 nm^-1 on GRID) of the given sources: from the per-field cache, the missing ones extracted from
    the ESA bulk files and calibrated with GaiaXPy, then added to the cache.'''
    xp_dir = 'xp_spectra_' + field_key
    os.makedirs(xp_dir, exist_ok=True)
    os.makedirs('xp_bulk_files', exist_ok=True)
    ids, F, E = load_xp_cache(xp_dir)
    missing = np.setdiff1d(source_ids, ids)
    if len(missing):
        from gaiaxpy import calibrate as xp_calibrate
        print('    XP: {:d} spectra to fetch for field {}'.format(len(missing), field_key), flush=True)
        tmp = xp_rows(missing)
        if tmp is not None:
            cal, _ = xp_calibrate(tmp, sampling=GRID, save_file=False)
            os.remove(tmp)
            np.savez_compressed(os.path.join(xp_dir, 'xp_{:d}.npz'.format(int(time.time()))),
                                source_id=cal['source_id'].values.astype(np.int64),
                                flux=np.vstack(cal['flux'].values).astype(np.float32),
                                flux_error=np.vstack(cal['flux_error'].values).astype(np.float32))
            print('    {:d} spectra calibrated'.format(len(cal)), flush=True)
        ids, F, E = load_xp_cache(xp_dir)
    keep = np.isin(ids, source_ids)
    return ids[keep], F[keep], E[keep]


def ab_mag(F, E, S):
    '''AB magnitude (and error) of the photon-weighted band flux of spectra F, E (rows, W m^-2 nm^-1 on GRID) through the response S.'''
    lam, dl = GRID * 10.0, DLAM * 10.0                                        # Angstrom
    w = S * lam * dl
    den = np.sum(S * dl / lam) * 2.99792458e18                                # c in Angstrom/s: f_nu = int(S f_lam lam dlam) / int(S c/lam dlam)
    f_nu = (F * 100.0) @ w / den                                              # 1 W m^-2 nm^-1 = 100 erg s^-1 cm^-2 A^-1
    e_nu = np.sqrt((E * 100.0) ** 2 @ w ** 2) / den
    with np.errstate(invalid='ignore', divide='ignore'):
        return -2.5 * np.log10(f_nu) - 48.60, 1.0857362 * e_nu / f_nu


def reference_mags(m, field_key, chip):
    '''The matched stars that have a published XP spectrum, with B_K_ref and B_K_ref_err through the chip passband.'''
    want = m.loc[m['has_xp_continuous'], 'source_id'].astype(np.int64).values
    if len(want) == 0:
        raise RuntimeError('no matched star has an XP spectrum -- check the has_xp_continuous column of the Gaia catalogue')
    ids, F, E = xp_spectra(want, field_key)
    xp = pd.DataFrame(dict(source_id=ids))
    xp['B_K_ref'], xp['B_K_ref_err'] = ab_mag(F, E, PASSBAND[chip]['S'])
    return m.merge(xp, on='source_id', how='inner')


def johnson_transformation(mx, stem):
    '''Coefficients t of  B_J - B_K = t0 + t1 dc + t2 dc^2  (dc = BP-RP - 1.5) between the KMTNet passband (AB) and GSPC Johnson B
    (Vega, Landolt-standardised): a passband transformation between two synthetic systems, fitted once to the 5000 brightest XP stars
    of one chip-frame that have validated GSPC magnitudes, cached in johnson_transformation.json, then applied everywhere.'''
    jf = 'johnson_transformation.json'
    if os.path.exists(jf) and json.load(open(jf)).get('version') == CFG['version']:
        return json.load(open(jf))
    g = mx[(mx['B_K_ref_err'] < 0.05) & mx['bp_rp'].between(*CFG['bp_rp_range'])].nsmallest(5000, 'phot_g_mean_mag')
    print('    Johnson transformation: querying the Gaia archive for the GSPC magnitudes of {:d} stars of {} (one job; can take minutes when the archive is busy) ...'.format(len(g), stem), flush=True)
    g = with_gspc(g, 'gspc_{}.csv'.format(stem))
    g = g[(g['b_jkc_flag'] == 1) & (g['b_jkc_mag_err'] < 0.05)]
    c = g['bp_rp'].values - REF
    f = robust_fit(np.vstack([np.ones_like(c), c, c ** 2]).T, (g['b_jkc_mag'] - g['B_K_ref']).values)
    jt = dict(t=f['p'].tolist(), sigma=float(f['sigma']), n=int(f['keep'].sum()), stem=stem, bp_rp_range=list(CFG['bp_rp_range']), version=CFG['version'])
    json.dump(jt, open(jf, 'w'), indent=1)
    return jt


def johnson_B(B_K, bp_rp, jt):
    '''Predicted Johnson B from B_K and the colour (clipped to the fitted range).'''
    c = np.clip(np.asarray(bp_rp, float), *jt['bp_rp_range']) - REF
    return B_K + jt['t'][0] + jt['t'][1] * c + jt['t'][2] * c ** 2

In [ ]:
# =============================================================================
# 2e. Calibration model and fit
#     MAG - B_K = a_0 + a_c dc + a_c2 dc^2 + sum_t a_t T_t(x, y) + a_amp(amplifier)
# =============================================================================

def design(bp_rp, X, Y, amp):
    '''Design matrix of the model. Columns, in the order of NAMES: 1; dc, dc^2 (dc = BP-RP - REF, NaN -> 0); the spatial terms in the
    chip coordinates normalised to [-1, 1] (even terms mean-subtracted over the chip, so that a_0 is the chip-averaged zero point);
    and sum-to-zero amplifier contrasts (amplifier j vs amplifier 0).'''
    c = np.nan_to_num(np.asarray(bp_rp, float) - REF)
    xn, yn = (np.asarray(X) - X0c) / X0c, (np.asarray(Y) - Y0c) / Y0c
    terms = dict(x=xn, y=yn, x2=xn * xn - 1.0 / 3.0, y2=yn * yn - 1.0 / 3.0, xy=xn * yn, x2y=xn * xn * yn, xy2=xn * yn * yn, y3=yn ** 3)
    a_ = np.asarray(amp)
    cols = ([np.ones_like(c)] + [c ** k for k in range(1, K + 1)] + [terms[t] for t in SP]
            + [(a_ == j).astype(float) - (a_ == 0).astype(float) for j in range(1, N_AMP)])
    return np.vstack(cols).T


def robust_fit(A, y):
    '''Least squares with iterative MAD-based sigma clipping. Returns p (coefficients), err (formal 1-sigma errors), keep (mask of
    the stars in the final fit), resid (all residuals) and sigma (robust scatter of the kept residuals).'''
    keep = np.isfinite(y) & np.all(np.isfinite(A), axis=1)
    for _ in range(CFG['clip_iters']):
        p = lstsq(A[keep], y[keep], rcond=None)[0]
        r = y - A @ p
        med = np.median(r[keep])
        mad = np.median(np.abs(r[keep] - med))
        new = keep & (np.abs(r - med) < CFG['clip_sigma'] * 1.4826 * max(mad, 1e-4))
        if new.sum() == keep.sum():
            break
        keep = new
    p = lstsq(A[keep], y[keep], rcond=None)[0]
    r = y - A @ p
    sig = 1.4826 * np.median(np.abs(r[keep] - np.median(r[keep])))
    try:
        cov = sig ** 2 * inv(A[keep].T @ A[keep])
    except np.linalg.LinAlgError:                                             # e.g. an amplifier stripe without calibration stars
        cov = sig ** 2 * np.linalg.pinv(A[keep].T @ A[keep])
    return dict(p=p, err=np.sqrt(np.abs(np.diag(cov))), keep=keep, resid=r, sigma=sig)


def calibration_sample(mx):
    '''The calibration stars: matched XP stars passing the DAOPHOT, Gaia and XP quality cuts of CFG, with offset = MAG - B_K_ref.'''
    mx['offset'] = mx['MAG'] - mx['B_K_ref']
    sel = ((mx['MERR'] < CFG['merr_max']) & (mx['CHI'] < CFG['chi_max']) & (np.abs(mx['SHARP']) < CFG['sharp_abs_max'])
           & (mx['MAG'] > CFG['mag_min']) & (mx['B_K_ref_err'] < CFG['bk_err_max']) & (np.abs(mx['c_star']) < CFG['cstar_abs_max'])
           & (mx['ruwe'] < CFG['ruwe_max']) & mx['phot_bp_mean_mag'].between(*CFG['bp_range'])
           & mx['bp_rp'].between(*CFG['bp_rp_range']) & np.isfinite(mx['offset']))
    return mx.loc[sel].copy()


class Calibration:
    '''A fitted model and how to apply it.'''

    def __init__(self, fit):
        self.p, self.err, self.sigma, self.keep, self.resid = fit['p'], fit['err'], fit['sigma'], fit['keep'], fit['resid']
        self.i_s = slice(1 + K, 1 + K + len(SP))                              # indices of the spatial coefficients
        self.i_g = slice(1 + K + len(SP), len(self.p))                        # indices of the amplifier contrasts
        self.amp_off = np.concatenate([[-self.p[self.i_g].sum()], self.p[self.i_g]])   # offsets of all N_AMP amplifiers (sum to zero)

    def zero_point(self, bp_rp, X, Y, amp):
        '''Model value MAG - B_K for the given colours and positions.'''
        return design(bp_rp, X, Y, amp) @ self.p

    def calibrate(self, frame, bp_rp):
        '''Calibrated B_K of the rows of frame (columns MAG, X, Y, amp) for the given colours (NaN -> reference colour).'''
        return frame['MAG'].values - self.zero_point(bp_rp, frame['X'].values, frame['Y'].values, frame['amp'].values)

    def spatial(self, X, Y):
        '''The spatial part of the model alone.'''
        X, Y = np.ravel(X), np.ravel(Y)
        pp = np.zeros_like(self.p)
        pp[self.i_s] = self.p[self.i_s]
        return design(np.full(len(X), REF), X, Y, np.zeros(len(X), int)) @ pp

    def correction_map(self, n=8):
        '''n x n map of the spatial + amplifier correction (zero point removed), for the comparison with the flat.'''
        cx, cy = np.meshgrid((np.arange(n) + 0.5) * CFG['image_width'] / n, (np.arange(n) + 0.5) * CFG['image_height'] / n)
        amp = (cx.ravel() // CFG['amp_width']).astype(int)
        return (self.zero_point(np.full(cx.size, REF), cx.ravel(), cy.ravel(), amp) - self.p[0]).reshape(n, n)


def binned(x, v, edges, min_n=10):
    '''Median of v in bins of x: rows of (bin centre, median, error of the median), bins with fewer than min_n points skipped.'''
    out = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        w = (x >= lo) & (x < hi)
        if w.sum() >= min_n:
            r = v[w]
            out.append((0.5 * (lo + hi), np.median(r), 1.2533 * 1.4826 * np.median(np.abs(r - np.median(r))) / np.sqrt(w.sum())))
    return np.array(out).reshape(-1, 3)


def cell_map(vals, X, Y, n=8):
    '''n x n map of the median of vals over the chip (NaN where a cell has fewer than 5 points); row index = Y.'''
    xe, ye = np.linspace(0, CFG['image_width'], n + 1), np.linspace(0, CFG['image_height'], n + 1)
    M = np.full((n, n), np.nan)
    for i in range(n):
        for j in range(n):
            w = (X >= xe[i]) & (X < xe[i + 1]) & (Y >= ye[j]) & (Y < ye[j + 1])
            if w.sum() >= 5:
                M[j, i] = np.median(vals[w])
    return M


def depth_metrics(df, matched_index):
    '''Depth of a calibrated chip: luminosity function of B_K (0.2-mag bins), its turnover (completeness limit) and half-maximum,
    the median DAOPHOT error vs B_K, and the fraction of stars without a Gaia counterpart per bin.'''
    B = df['B_K'].values
    bins = np.arange(8, 24.01, 0.2)
    mids = 0.5 * (bins[1:] + bins[:-1])
    lf, _ = np.histogram(B, bins)
    ipk = int(np.argmax(lf))
    faint = np.where((mids > mids[ipk]) & (lf < 0.5 * lf[ipk]))[0]
    is_m = np.zeros(len(df), bool)
    is_m[matched_index] = True
    ib = np.clip(np.digitize(B, bins) - 1, 0, len(mids) - 1)
    unmatched = np.array([1 - is_m[ib == i].mean() if (ib == i).sum() >= 10 else np.nan for i in range(len(mids))])
    merr = [(float(mm), float(np.median(df.loc[np.abs(B - mm) < 0.25, 'MERR'])))
            for mm in np.arange(12, 22.1, 0.5) if (np.abs(B - mm) < 0.25).sum() >= 20]
    return dict(B_turnover=float(mids[ipk]), B_half=float(mids[faint[0]]) if len(faint) else np.nan, lf_B=lf.tolist(), lf_mids=mids.tolist(),
                unmatched_frac=unmatched.tolist(), unmatched_at_B20=float(unmatched[np.argmin(np.abs(mids - 20.0))]), merr_vs_B=merr)

In [ ]:
# =============================================================================
# 2f. One chip-frame end to end
# =============================================================================

def prepare_frame(cat, csv, chip):
    '''Everything up to the fit for one (frame, chip): catalogue, astrometry, Gaia match, reference magnitudes and calibration
    sample -- all from the caches when they exist. Returns a dict with the pieces.'''
    meta = frame_meta(frame_id(csv), chip)
    meta['stem'] = stem_of(csv, chip)
    df = chip_catalogue(cat, chip)
    astrom = solve_wcs(df, meta)
    gdf, field_key = gaia_field(df)
    m, mstats = match_gaia(df, gdf, meta['epoch'])
    if mstats['match_frac'] < CFG['min_match_frac']:                        # a poor solution: refine it once more against Gaia
        hf = meta['stem'] + '_anet_wcs.hdr'
        hdr, info = refine_wcs_with_gaia(df, WCS(fits.Header.fromtextfile(hf)), gdf, 'poor solution')
        if refined_ok(hdr, info):
            hdr.totextfile(hf, overwrite=True)
            astrom = apply_wcs(df, hdr)
            m, mstats = match_gaia(df, gdf, meta['epoch'])
        print('    {}: {:.0%} of the stars matched after re-refinement'.format(meta['stem'], mstats['match_frac']), flush=True)
    meta['astrometry_suspect'] = bool(mstats['match_frac'] < CFG['min_match_frac'])
    gal = SkyCoord(astrom['ra_c'] * u.deg, astrom['dec_c'] * u.deg).galactic
    meta.update(l=float(gal.l.wrap_at(180 * u.deg).deg), b=float(gal.b.deg))
    mx = reference_mags(m, field_key, chip)
    return dict(df=df, meta=meta, astrom=astrom, gdf=gdf, field_key=field_key, m=m, mstats=mstats, mx=mx, hs=calibration_sample(mx))


def star_colours(m, cal):
    '''The BP-RP colour used to calibrate every matched star: Gaia's where it exists; otherwise inferred from B_K - G through a cubic
    relation fitted to the stars with both (three fixed-point passes, the colour entering B_K only through the small colour term).
    Returns the colours, their source ('bp_rp', 'B-G', 'assumed'), and the colour assumed for stars without any Gaia counterpart
    (median colour of the fainter half of the matched stars).'''
    c = m['bp_rp'].values.copy()
    src = np.where(np.isfinite(c), 'bp_rp', 'none').astype(object)
    G = m['phot_g_mean_mag'].values
    has_c, has_g = np.isfinite(c), np.isfinite(G)
    bg = cal.calibrate(m, c) - G
    ok = has_c & has_g & np.isfinite(bg)
    if ok.sum() > 200:
        rel = np.polyfit(bg[ok], c[ok], 3)
        need = ~has_c & has_g
        lo, hi = CFG['bp_rp_range']
        c[need] = np.clip(np.polyval(rel, bg[need]), lo, hi)
        for _ in range(3):
            c[need] = np.clip(np.polyval(rel, cal.calibrate(m, c)[need] - G[need]), lo, hi)
        src[need] = 'B-G'
    faint = has_c & (G > np.nanmedian(G))
    assumed = float(np.nanmedian(c[faint])) if faint.sum() > 50 else REF
    still = ~np.isfinite(c)
    c[still] = assumed
    src[still] = 'assumed'
    return c, src, assumed


def write_catalogue(df, m, mx, hs, fit, stem):
    '''The calibrated catalogue of every DAOPHOT detection: B_K, B_J_pred, the colour used, and its Gaia match (NaN / empty if none).
    Written as csv and, if write_fits, as FITS. Returns the DataFrame.'''
    out = df[['ID', 'X', 'Y', 'amp', 'RA', 'DEC', 'MAG', 'MERR', 'MSKY', 'NITER', 'CHI', 'SHARP', 'B_K', 'B_J_pred', 'bp_rp_used', 'color_source']].copy()
    out['B_K_err'] = df['MERR']
    out['matched'] = 0
    out['calib_star'] = 0
    for col in ['sep_arcsec', 'B_K_ref', 'B_K_ref_err'] + GAIA_COLS:
        out[col] = '' if col in STRING_COLS else np.nan
    pos = m['df_index'].values                                                # rows of df with a Gaia match
    out.loc[pos, 'matched'] = 1
    out.loc[pos, 'sep_arcsec'] = m['sep_arcsec'].values
    for col in GAIA_COLS:
        out.loc[pos, col] = m[col].astype(str).values if col in STRING_COLS else m[col].astype(float).values
    out.loc[mx['df_index'].values, ['B_K_ref', 'B_K_ref_err']] = mx[['B_K_ref', 'B_K_ref_err']].values
    out.loc[hs['df_index'].values[fit['keep']], 'calib_star'] = 1
    for col in ('source_id', 'astrometric_params_solved', 'has_xp_continuous', 'matched', 'calib_star', 'amp'):
        out[col] = out[col].astype('Int64')
    out['B_minus_G'] = out['B_K'] - out['phot_g_mean_mag']
    out.to_csv(stem + '_calibrated_catalog.csv', index=False)
    if CFG['write_fits']:
        Table.from_pandas(out.astype({c: str for c in STRING_COLS + ('color_source',)})).write(stem + '_calibrated_catalog.fits', overwrite=True)
    return out


def run_frame(cat, csv, chip, make_figure=True):
    '''Calibrate one (frame, chip): fit the model, calibrate every detection, write the catalogue, the JSON record and the diagnostic
    figure. Returns the record.'''
    t0 = time.time()
    P = prepare_frame(cat, csv, chip)
    df, meta, m, mx, hs = P['df'], P['meta'], P['m'], P['mx'], P['hs']
    stem = meta['stem']
    if len(hs) < CFG['min_calib_stars']:
        raise RuntimeError('{}: only {:d} calibration stars'.format(stem, len(hs)))
    # the fit
    A = design(hs['bp_rp'], hs['X'], hs['Y'], hs['amp'])
    fit = robust_fit(A, hs['offset'].values)
    cal = Calibration(fit)
    # y-gradient of the bright and faint halves: a CTI-like effect is signal dependent, a flat-field one is not
    G = hs['phot_g_mean_mag'].values
    gmed = np.median(G)
    sy = [float(robust_fit(A[w], hs['offset'].values[w])['p'][NAMES.index('s_y')]) for w in (G <= gmed, G > gmed)]
    # calibrated magnitudes of every star
    jt = JT
    c, src, assumed = star_colours(m, cal)
    df['bp_rp_used'], df['color_source'] = assumed, 'assumed'
    df.loc[m['df_index'].values, 'bp_rp_used'] = c
    df.loc[m['df_index'].values, 'color_source'] = src
    df['B_K'] = cal.calibrate(df, df['bp_rp_used'].values)
    df['B_J_pred'] = johnson_B(df['B_K'].values, df['bp_rp_used'].values, jt)
    out = write_catalogue(df, m, mx, hs, fit, stem)
    # the record
    grid = np.meshgrid(np.linspace(0, CFG['image_width'], 41), np.linspace(0, CFG['image_height'], 41))
    rec = dict(version=CFG['version'], frame=meta['frame'], stem=stem, chip=chip, object=meta['object'], field=P['field_key'],
               l=meta['l'], b=meta['b'], airmass=meta['airmass'], date_obs=meta['date_obs'], epoch=meta['epoch'], exptime=meta['exptime'],
               astrometry=P['astrom'], astrometry_suspect=meta['astrometry_suspect'], match=P['mstats'],
               names=NAMES, values=cal.p.tolist(), err_formal=cal.err.tolist(), sigma_mad=float(cal.sigma), n_calib=int(fit['keep'].sum()),
               amp_offsets=cal.amp_off.tolist(), sy_bright=sy[0], sy_faint=sy[1], spatial_p2p=float(np.ptp(cal.spatial(*grid))),
               correction_map_8x8=cal.correction_map().tolist(), depth=depth_metrics(df, m['df_index'].values),
               color_source_counts={k: int((df['color_source'] == k).sum()) for k in ('bp_rp', 'B-G', 'assumed')}, color_assumed=assumed,
               johnson_transformation=jt['t'], runtime_s=time.time() - t0)
    json.dump(rec, open(stem + '_calibration.json', 'w'), indent=1)
    if make_figure:
        frame_figure(rec, hs, cal, out, stem)
    return rec


def frame_figure(rec, hs, cal, out, stem):
    '''Six diagnostic panels of one chip-frame: residual vs colour, vs magnitude, residual maps before/after the spatial terms,
    amplifier stripes, and the Gaia-KMTNet CMD. Saved to figs/<stem>_diagnostics.png.'''
    fig, ax = plt.subplots(2, 3, figsize=(21, 12))
    resid = cal.resid
    Xs, Ys = hs['X'].values, hs['Y'].values
    # (0, 0): residual vs colour
    b = binned(hs['bp_rp'].values, resid, CFG['color_bins'])
    ax[0, 0].errorbar(b[:, 0], b[:, 1], yerr=b[:, 2], fmt='o-', color='k', ms=4, capsize=2)
    ax[0, 0].set(ylim=(-0.05, 0.05), xlabel='Gaia BP - RP', ylabel='residual [mag]', title='residual by colour (c1 {:+.4f}, c2 {:+.4f})'.format(cal.p[1], cal.p[2]))
    # (0, 1): residual vs magnitude
    x = hs['phot_g_mean_mag'].values
    ax[0, 1].scatter(x, resid, s=2, color='0.75', alpha=0.5)
    b = binned(x, resid, np.arange(np.floor(x.min()), x.max() + 0.25, 0.25))
    ax[0, 1].errorbar(b[:, 0], b[:, 1], yerr=b[:, 2], fmt='o-', color='k', ms=4, capsize=2)
    ax[0, 1].set(ylim=(-0.2, 0.2), xlabel='Gaia G', ylabel='residual [mag]', title='residual vs magnitude (sigma {:.4f})'.format(cal.sigma))
    # (0, 2) and (1, 0): residual maps of a zero-point + colour-term fit and of the full model
    A_c = design(hs['bp_rp'].values, Xs, Ys, hs['amp'].values)[:, :1 + K]
    y = hs['offset'].values
    r0 = y - A_c @ lstsq(A_c[cal.keep], y[cal.keep], rcond=None)[0]
    M0, M1 = cell_map(r0, Xs, Ys), cell_map(resid, Xs, Ys)
    vmax = max(0.02, np.nanmax(np.abs(M0)))
    for axx, M, ttl in ((ax[0, 2], M0, 'residual map, colour-only fit'), (ax[1, 0], M1, 'residual map, full model')):
        im = axx.imshow(M, origin='lower', extent=[0, CFG['image_width'], 0, CFG['image_height']], cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        plt.colorbar(im, ax=axx, label='mag')
        axx.set(title=ttl, xlabel='X', ylabel='Y')
    # (1, 1): residual of the colour-only fit along X in four Y bands, with the fitted amplifier offsets
    xe = np.arange(0, CFG['image_width'] + 96, 96)
    xm = 0.5 * (xe[1:] + xe[:-1])
    ye = np.linspace(0, CFG['image_height'], 5)
    for k in range(4):
        wy = (Ys >= ye[k]) & (Ys < ye[k + 1])
        prof = [np.median(r0[wy & (Xs >= lo) & (Xs < hi)]) if (wy & (Xs >= lo) & (Xs < hi)).sum() >= 8 else np.nan for lo, hi in zip(xe[:-1], xe[1:])]
        ax[1, 1].plot(xm, prof, '-', lw=1.1, label='Y {:.0f}-{:.0f}'.format(ye[k], ye[k + 1]))
    for j in range(1, N_AMP):
        ax[1, 1].axvline(j * CFG['amp_width'], color='0.6', ls=':', lw=0.8)
    ax[1, 1].plot((np.arange(N_AMP) + 0.5) * CFG['amp_width'], cal.amp_off, 'ks', ms=6, label='fitted amplifier offsets')
    ax[1, 1].legend(fontsize=11, ncol=2)
    ax[1, 1].set(ylim=(-0.12, 0.12), xlabel='X [pix]', ylabel='residual, colour-only fit [mag]', title='amplifier stripes')
    # (1, 2): CMD of the matched stars
    ok = np.isfinite(out['B_minus_G'].values) & np.isfinite(out['phot_g_mean_mag'].values.astype(float))
    xg, g = out.loc[ok, 'B_minus_G'].values, out.loc[ok, 'phot_g_mean_mag'].values.astype(float)
    ax[1, 2].hist2d(xg, g, bins=120, range=[np.percentile(xg, [0.2, 99.8]), (g.min() - 0.2, g.max() + 0.2)], norm=LogNorm(), cmap='magma')
    ax[1, 2].invert_yaxis()
    ax[1, 2].set(xlabel='B_K - G', ylabel='Gaia G', title='CMD, {:d} matched stars'.format(int(ok.sum())))
    for a_ in (ax[0, 0], ax[0, 1], ax[1, 1]):
        a_.axhline(0, color='k', lw=0.8)
    plt.suptitle('{}   {}   (l, b) = ({:+.2f}, {:+.2f})   chip {}   X = {:.2f}'.format(rec['frame'], rec['object'], rec['l'], rec['b'], rec['chip'], rec['airmass']), y=1.0)
    plt.tight_layout()
    fig.savefig('figs/{}_diagnostics.png'.format(stem), dpi=90, bbox_inches='tight')
    if CFG['show_frame_figures']:
        plt.show()
    else:
        plt.close(fig)


print('pipeline functions defined')

## 2b. Johnson transformation
$B_J - B_{\rm K}$ as a function of $G_{\rm BP}-G_{\rm RP}$, fitted once between the synthetic $B_{\rm K}$ and the GSPC Johnson $B$ of the
same XP stars (first chip-frame of the program, one Gaia-archive job) and cached in `johnson_transformation.json`; delete that file to
refit, e.g. on another frame processed first.

In [ ]:
# the transformation is fitted once (first frame, first chip) and cached; later runs only load it
if os.path.exists('johnson_transformation.json') and json.load(open('johnson_transformation.json')).get('version') == CFG['version']:
    JT = json.load(open('johnson_transformation.json'))
else:                                                                        # absent, or fitted with another passband definition
    csv_j, chip_j = FRAME_FILES[0], CFG['chips'][0]
    JT = johnson_transformation(prepare_frame(load_frame(csv_j), csv_j, chip_j)['mx'], stem_of(csv_j, chip_j))
print('B_J - B_K = {:+.4f} {:+.4f} dc {:+.4f} dc^2   (sigma {:.3f} mag, {:d} stars of {})'.format(*JT['t'], JT['sigma'], JT['n'], JT['stem']))

## 3. Batch: every frame, every chip
Resumable: a (frame, chip) whose record was written by this notebook version is only loaded (seconds for the whole program), so
re-running the notebook for the later sections costs nothing here. Failures are collected and listed at the end rather than stopping
the batch.

In [ ]:
def current_record(stem):
    '''The cached record of a chip-frame if it was written by this notebook version, is not flagged, and its catalogue exists.'''
    jf = stem + '_calibration.json'
    if not os.path.exists(jf):
        return None
    r = json.load(open(jf))
    ok = r.get('version') == CFG['version'] and not r.get('astrometry_suspect') and os.path.exists(stem + '_calibrated_catalog.csv')
    return r if ok else None


records, failures, t_batch = {}, [], time.time()
items = [(csv, chip) for csv in FRAME_FILES for chip in CFG['chips'] if stem_of(csv, chip) not in CFG['skip_items']]
n_cached = sum(current_record(stem_of(c, ch)) is not None for c, ch in items)
print('{:d} chip-frames; {:d} already have a current record'.format(len(items), n_cached), flush=True)
for k_f, csv in enumerate(FRAME_FILES):
    cat = None                                                                # the four-chip file is read once per frame, when needed
    for chip in CFG['chips']:
        stem = stem_of(csv, chip)
        if stem in CFG['skip_items']:
            continue
        rec = current_record(stem)
        src = 'cached'
        if rec is None:
            try:
                cat = load_frame(csv) if cat is None else cat
                rec = run_frame(cat, csv, chip)
                src = '{:.0f} s'.format(rec['runtime_s'])
            except Exception as e:
                failures.append((stem, str(e)[:120]))
                print('{:40s} FAILED: {}'.format(stem, str(e)[:100]), flush=True)
                continue
        records[stem] = rec
        d, mt, cs = rec['depth'], rec['match'], rec['color_source_counts']
        print('{:40s} {} ({:+.1f},{:+.1f})  N={:6d} matched={:.1%} astrom={:.3f}"  Ncal={:5d} sigma={:.4f} p2p={:.3f}  a_c={:+.4f}  B_turn={:.1f}  colours {:d}/{:d}/{:d}  [{}]{}'.format(
              stem, rec['object'], rec['l'], rec['b'], mt['n_kmt'], mt['match_frac'], mt['astrom_sigma'], rec['n_calib'], rec['sigma_mad'],
              rec['spatial_p2p'], rec['values'][1], d['B_turnover'], cs['bp_rp'], cs['B-G'], cs['assumed'], src,
              '  ** ASTROMETRY SUSPECT **' if rec['astrometry_suspect'] else ''), flush=True)
    print('  -- frame {:d}/{:d} done, {:.0f} min elapsed'.format(k_f + 1, len(FRAME_FILES), (time.time() - t_batch) / 60), flush=True)
print('\n{:d} records; {:d} failures'.format(len(records), len(failures)))
for s, msg in failures:
    print('  ', s, ':', msg)

## 4. Program summary
One row per (frame, chip) with every scalar quantity is written to `calibration_summary.csv`; per-chip medians and frame-to-frame
scatters of the key quantities are printed, and the sky coverage, colour term and amplifier offsets are plotted.

In [ ]:
def rec_row(r):
    '''One summary row (every scalar of a record): metadata, match and depth statistics, all coefficients with their formal errors,
    and the amplifier offsets amp1..amp8.'''
    d, mt, cs = r['depth'], r['match'], r['color_source_counts']
    row = dict(stem=r['stem'], frame=r['frame'], chip=r['chip'], object=r['object'], field=r['field'], l=r['l'], b=r['b'],
               airmass=r['airmass'], date=r['date_obs'], n_kmt=mt['n_kmt'], n_matched=mt['n_matched'], match_frac=mt['match_frac'],
               astrom_sigma=mt['astrom_sigma'], n_calib=r['n_calib'], sigma_mad=r['sigma_mad'], spatial_p2p=r['spatial_p2p'],
               B_turnover=d['B_turnover'], B_half=d['B_half'], unmatched_at_B20=d['unmatched_at_B20'],
               frac_BG=cs['B-G'] / mt['n_kmt'], frac_assumed=cs['assumed'] / mt['n_kmt'], sy_bright=r['sy_bright'], sy_faint=r['sy_faint'],
               pixscale=r['astrometry']['pixscale'])
    row.update(zip(r['names'], r['values']))
    row.update({n + '_err': v for n, v in zip(r['names'], r['err_formal'])})
    row.update({'amp{:d}'.format(j + 1): v for j, v in enumerate(r['amp_offsets'])})
    return row


SUM = pd.DataFrame([rec_row(r) for r in records.values()]).set_index('stem').sort_values(['frame', 'chip'])
SUM.to_csv('calibration_summary.csv')
pd.set_option('display.width', 260)
pd.set_option('display.max_columns', 80)
key = ['n_kmt', 'match_frac', 'astrom_sigma', 'n_calib', 'sigma_mad', 'spatial_p2p', 'B_turnover', 'a', 'c1', 'c2', 's_y']
print('Per-chip medians (and frame-to-frame scatter) over {:d} records:'.format(len(SUM)))
tab = pd.concat({'median': SUM.groupby('chip')[key].median(), 'scatter': SUM.groupby('chip')[key].std()}, axis=1)
print(tab.swaplevel(axis=1).sort_index(axis=1, level=0).to_string(float_format=lambda v: '{:.4f}'.format(v)))
print('\nAll chips: sigma_mad median {:.4f} (range {:.4f}..{:.4f});  astrometric scatter median {:.3f}";  matched fraction {:.1%}..{:.1%};  B turnover {:.1f}..{:.1f}'.format(
      SUM['sigma_mad'].median(), SUM['sigma_mad'].min(), SUM['sigma_mad'].max(), SUM['astrom_sigma'].median(),
      SUM['match_frac'].min(), SUM['match_frac'].max(), SUM['B_turnover'].min(), SUM['B_turnover'].max()))
print('Johnson transformation applied: B_J - B_K = {:+.4f} {:+.4f} dc {:+.4f} dc^2'.format(*next(iter(records.values()))['johnson_transformation']))

# overview figure: sky map coloured by the residual scatter; colour term vs stellar density; amplifier offsets per chip
CHIP_COLORS = dict(K='C0', M='C1', N='C2', T='C3')
xs = np.arange(1, N_AMP + 1)
fig, ax = plt.subplots(1, 3, figsize=(21, 5.5))
sc = ax[0].scatter(SUM['l'], SUM['b'], c=SUM['sigma_mad'], s=60, marker='s', cmap='viridis', edgecolor='k', lw=0.3)
ax[0].invert_xaxis()
ax[0].set(xlabel='l [deg]', ylabel='b [deg]', title='chip centres, coloured by residual scatter')
plt.colorbar(sc, ax=ax[0], label='sigma_MAD [mag]')
for ch, g in SUM.groupby('chip'):
    ax[1].scatter(g['n_kmt'], g['c1'], color=CHIP_COLORS[ch], s=30, label='chip ' + ch)
    v = g[['amp{:d}'.format(j) for j in xs]].values
    ax[2].errorbar(xs + 0.1 * (list(CFG['chips']).index(ch) - 1.5), np.median(v, axis=0), yerr=np.std(v, axis=0), fmt='o-',
                   color=CHIP_COLORS[ch], ms=6, capsize=3, label='chip ' + ch)
ax[1].set(xscale='log', xlabel='KMTNet detections per chip', ylabel='$a_c$ [mag/mag]', title='colour term vs stellar density')
ax[1].legend()
ax[2].axhline(0, color='k', lw=0.8)
ax[2].set(xticks=xs, xlabel='amplifier', ylabel='$a_{\\rm amp}$: median and scatter over frames [mag]', title='amplifier offsets per chip')
ax[2].legend()
plt.tight_layout()
fig.savefig('figs/program_overview.png', dpi=90, bbox_inches='tight')
plt.show()

## 5. Repeated pointings
Frames of the same field and chip observed at different times: the zero point should differ (transparency, airmass), the instrument
terms should not. Differences are listed in magnitudes; their rms over pairs measures the night-to-night stability.

In [ ]:
cols = ['a', 'c1', 'c2', 's_y', 's_y2', 's_xy', 'g_02', 'g_08']
pairs = []
for (fld, ch), g in SUM.groupby(['field', 'chip']):
    g = g.sort_values('date')
    base = g.iloc[0]
    for _, r in g.iloc[1:].iterrows():
        pairs.append(dict(field=fld, chip=ch, frames='{} -> {}'.format(base['frame'][-6:], r['frame'][-6:]), **{c: r[c] - base[c] for c in cols}))
if pairs:
    P = pd.DataFrame(pairs)
    print('Differences between repeated observations (later minus earlier), {:d} pairs:'.format(len(P)))
    print(P.to_string(index=False, float_format=lambda v: '{:+.4f}'.format(v)))
    print('\n  rms over pairs: ' + '  '.join('{} {:.4f}'.format(c, P[c].std(ddof=0)) for c in cols))
else:
    print('No repeated pointings in this set.')

## 6. Master sky flat versus the spatial correction, per chip
The mean total correction map of each chip (over its frames) is decomposed into the part that follows the flat's large-scale structure
($+2.5\log_{10}$ of the 8×8 cell medians) and a remainder.

In [ ]:
if CFG['flat_file'] and os.path.exists(CFG['flat_file']):
    n = 8
    fig, ax = plt.subplots(len(CFG['chips']), 3, figsize=(16, 4.6 * len(CFG['chips'])), squeeze=False)
    with fits.open(CFG['flat_file']) as hdul:                                  # the flat of a chip is its 8 amplifier stripes side by side
        flats = {}
        for ch in CFG['chips']:
            stripes = sorted([x for x in hdul[1:] if x.name.upper().startswith(ch)], key=lambda x: x.name)
            if stripes:
                flats[ch] = np.hstack([np.asarray(x.data, float) for x in stripes])
    for k_c, ch in enumerate(CFG['chips']):
        maps = [np.array(r['correction_map_8x8']) for r in records.values() if r['chip'] == ch]
        if ch not in flats or not maps:
            print('chip {}: no flat extensions or no records'.format(ch))
            continue
        flat = flats[ch]
        ny, nx = flat.shape
        Fm = np.array([[np.nanmedian(flat[j * ny // n:(j + 1) * ny // n, i * nx // n:(i + 1) * nx // n]) for i in range(n)] for j in range(n)])
        flat_mag = 2.5 * np.log10(Fm / np.nanmedian(Fm))                     # large-scale structure of the flat, in magnitudes
        corr = np.mean(maps, axis=0)
        corr -= corr.mean()                                                   # mean stellar correction map of the chip
        alpha, ic = np.polyfit(flat_mag.ravel(), corr.ravel(), 1)             # correction = alpha x flat + remainder
        cc = np.corrcoef(flat_mag.ravel(), corr.ravel())[0, 1]
        rem = corr - (alpha * flat_mag + ic)
        print('chip {}: {:d} frames; correction = {:+.2f} x flat + remainder, r = {:+.2f}, variance explained {:.0f}%; remainder y-gradient {:+.3f} mag, x-gradient {:+.3f} mag'.format(
              ch, len(maps), alpha, cc, 100 * (1 - rem.var() / corr.var()), rem.mean(axis=1)[-1] - rem.mean(axis=1)[0], rem.mean(axis=0)[-1] - rem.mean(axis=0)[0]))
        v = max(0.02, np.abs(np.concatenate([flat_mag.ravel(), corr.ravel()])).max())
        panels = ((ax[k_c, 0], flat_mag, 'chip {}: +2.5 log10(flat)'.format(ch)), (ax[k_c, 1], corr, 'mean stellar correction ({:d} frames)'.format(len(maps))),
                  (ax[k_c, 2], rem, 'remainder (alpha = {:+.2f}, r = {:+.2f})'.format(alpha, cc)))
        for axx, M, ttl in panels:
            im = axx.imshow(M, origin='lower', extent=[0, nx, 0, ny], cmap='RdBu_r', vmin=-v, vmax=v, aspect='equal')
            plt.colorbar(im, ax=axx, label='mag')
            axx.set(title=ttl, xlabel='X', ylabel='Y')
    plt.tight_layout()
    plt.show()
else:
    print('No flat file; skipping')

## 7. Figures for the paper
Journal PDFs in `figs/paper_*.pdf` (raster-heavy elements rasterized). `paper_frame`/`paper_chip` select the single-frame panels,
`paper_cmd_items` the (frame, chip) pairs of the CMD and depth figures. The Roman GBTDS footprint (Penny, `gbtds_optimizer`,
layouts `gbtds_{spring,autumn}_2026.4.3`, downloaded once into `gbtds/`) is drawn on the coverage map and used in Section 8.

In [ ]:
# ---- choices for the paper figures ---------------------------------------------------------------------------------------------------
paper_frame, paper_chip = 'xkmts.20230715.020746', 'T'      # single-frame panels (BLG01 chip T); paper_frame = None -> the paper_chip record with the most calibration stars
paper_cmd_items = [('xkmts.20230715.020733', 'K'), ('xkmts.20230715.020746', 'T'), ('xkmts.20230715.020754', 'T'), ('xkmts.20230716.021073', 'T')]   # sparse to dense
COL1, COL2 = 3.5, 7.2                                        # column widths [inch]
PAPER_RC = {'font.family': 'serif', 'font.size': 9, 'axes.labelsize': 10, 'axes.titlesize': 10, 'legend.fontsize': 8, 'xtick.labelsize': 8,
            'ytick.labelsize': 8, 'savefig.dpi': 200, 'pdf.fonttype': 42, 'axes.linewidth': 0.8}
SCREEN_RC = {k: plt.rcParams[k] for k in PAPER_RC}
plt.rcParams.update(PAPER_RC)


def savepdf(fig, name):
    '''Save figs/paper_<name>.pdf, show the figure and report the file size.'''
    f = 'figs/paper_{}.pdf'.format(name)
    fig.savefig(f, bbox_inches='tight', pad_inches=0.02)
    plt.show()
    print('  wrote {} ({:.0f} kB)'.format(f, os.path.getsize(f) / 1e3))


def rec_of(frame, chip):
    '''The record of one (frame, chip).'''
    return next(r for r in records.values() if r['frame'] == frame and r['chip'] == chip)


def inward(*axes):
    '''Ticks inward on all four sides.'''
    for a_ in axes:
        a_.tick_params(direction='in', top=True, right=True)


# ---- Roman GBTDS footprint and KMTNet chip footprints in Galactic coordinates ------------------------------------------------------
GBTDS_RAW = 'https://raw.githubusercontent.com/mtpenny/gbtds_optimizer/7c2e5e5ec25d41183117769f51a491c4012c57c6/'
GBTDS_FILES = dict(spring=('field_layouts/gbtds_spring_2026.4.3.centers', 'outline_sca_layout_spring.txt'),
                   autumn=('field_layouts/gbtds_autumn_2026.4.3.centers', 'outline_sca_layout_autumn.txt'))


def gbtds_footprints():
    '''[(season, field, Nx2 (l, b) outline)] of the Roman GBTDS fields, both seasons (Penny, gbtds_optimizer, layout 2026.4.3; the files
    are downloaded once into gbtds/).'''
    os.makedirs('gbtds', exist_ok=True)
    out = []
    for season, files in GBTDS_FILES.items():
        for f in files:
            local = os.path.join('gbtds', os.path.basename(f))
            if not os.path.exists(local):
                open(local, 'wb').write(requests.get(GBTDS_RAW + f, timeout=60).content)
        cen = pd.read_csv(os.path.join('gbtds', os.path.basename(files[0])), sep=r'\s+')
        outline = pd.read_csv(os.path.join('gbtds', os.path.basename(files[1])), sep=r'\s+', header=None)
        out += [(season, str(r['field']), np.c_[outline[1].values + r['l'], outline[2].values + r['b']]) for _, r in cen.iterrows()]
    return out


GBTDS = gbtds_footprints()


def chip_footprint_lb(stem):
    '''Corners of the chip in Galactic coordinates (4x2) from its cached WCS.'''
    w = WCS(fits.Header.fromtextfile(stem + '_anet_wcs.hdr'))
    W, H = CFG['image_width'], CFG['image_height']
    g = SkyCoord(*w.all_pix2world([0.5, W + 0.5, W + 0.5, 0.5], [0.5, 0.5, H + 0.5, H + 0.5], 1), unit='deg').galactic
    return np.c_[g.l.wrap_at(180 * u.deg).deg, g.b.deg]


def in_gbtds(poly):
    '''True if the (l, b) polygon overlaps any GBTDS field outline of either season.'''
    p = Path(np.vstack([poly, poly[:1]]), closed=True)
    return any(p.intersects_path(Path(fp, closed=True), filled=True) for _, _, fp in GBTDS)


FOOTPRINT = {stem: chip_footprint_lb(stem) for stem in SUM.index}
SUM['gbtds'] = [in_gbtds(FOOTPRINT[s]) for s in SUM.index]
print('Chip-frames overlapping the Roman GBTDS footprint: {:d} of {:d}; fields: {}'.format(
      int(SUM['gbtds'].sum()), len(SUM), ', '.join(sorted(SUM.loc[SUM['gbtds'], 'object'].unique()))))

# ---- the single frame of Figs. 2 and 3: its calibration sample and fit, re-derived from the caches ------------------------------------
cands = SUM[SUM['chip'] == paper_chip]
if paper_frame is None:
    paper_frame = cands.sort_values('n_calib').iloc[-1]['frame']
rec0 = rec_of(paper_frame, paper_chip)
csv0 = next(f for f in FRAME_FILES if frame_id(f) == paper_frame)
P0 = prepare_frame(load_frame(csv0), csv0, paper_chip)
hs0 = P0['hs']
A0 = design(hs0['bp_rp'], hs0['X'], hs0['Y'], hs0['amp'])
fit0 = robust_fit(A0, hs0['offset'].values)
cal0 = Calibration(fit0)
print('Single-frame panels use {} chip {} ({}, {:d} calibration stars); CMD / depth figures use {}'.format(
      paper_frame, paper_chip, rec0['object'], int(fit0['keep'].sum()), paper_cmd_items))

# ---- Fig. 1: the system passband --------------------------------------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(COL1, 3.1))
ax.plot(GRID, S_FILTER, color='0.15', lw=1.2, label='filter (lab, mean of 9 positions)')
ax.plot(GRID, QE_MEAN, color='crimson', lw=1.0, ls='--', label='CCD quantum efficiency (mean of the 4 chips)')
ax.plot(GRID, S_SYS, color='C0', lw=1.8, label='system passband (filter $\\times$ QE, {:.0f} nm)'.format(LAM_EFF))
ax.set(xlim=(360, 545), ylim=(0, 1.02), xlabel='wavelength [nm]', ylabel='response / transmission')
inward(ax)
ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.01), ncol=1, frameon=False, fontsize=7, handlelength=2.2)
savepdf(fig, 'passband')

# ---- Fig. 2: calibration residuals of one frame (1 x 3) -------------------------------------------------------------------------------
resid0, Xs, Ys = cal0.resid, hs0['X'].values, hs0['Y'].values
A_c = A0[:, :1 + K]                                                            # zero point + colour terms only
y0 = hs0['offset'].values
r_col = y0 - A_c @ lstsq(A_c[fit0['keep']], y0[fit0['keep']], rcond=None)[0]
M0, M1 = cell_map(r_col, Xs, Ys), cell_map(resid0, Xs, Ys)
mad_cells = lambda M: 1.4826 * np.nanmedian(np.abs(M - np.nanmedian(M)))
print('  8x8 cell-median residual map: robust scatter {:.4f} -> {:.4f} mag, peak-to-peak {:.3f} -> {:.3f} mag (zero point + colour term only -> full model)'.format(
      mad_cells(M0), mad_cells(M1), np.nanmax(M0) - np.nanmin(M0), np.nanmax(M1) - np.nanmin(M1)))
fig, ax = plt.subplots(1, 3, figsize=(COL2, 2.6), constrained_layout=True)
b = binned(hs0['bp_rp'].values, resid0, CFG['color_bins'])
ax[0].errorbar(b[:, 0], b[:, 1], yerr=b[:, 2], fmt='o-', color='k', ms=3, lw=1, capsize=2)
ax[0].axhline(0, color='0.5', lw=0.6)
ax[0].set(ylim=(-0.04, 0.04), xlabel='Gaia $(G_{\\rm BP}-G_{\\rm RP})$', ylabel='median residual [mag]')
x = hs0['phot_g_mean_mag'].values
ax[1].scatter(x, resid0, s=1.5, color='0.7', alpha=0.4, rasterized=True)
b = binned(x, resid0, np.arange(np.floor(x.min()), x.max() + 0.25, 0.25))
ax[1].errorbar(b[:, 0], b[:, 1], yerr=b[:, 2], fmt='o-', color='k', ms=3, lw=1, capsize=2)
ax[1].axhline(0, color='0.5', lw=0.6)
ax[1].set(ylim=(-0.15, 0.15), xlabel='Gaia $G$', ylabel='residual [mag]')
vmax = max(0.02, np.nanmax(np.abs(M1)))
im = ax[2].imshow(M1, origin='lower', extent=[0, CFG['image_width'], 0, CFG['image_height']], cmap='RdBu_r', vmin=-vmax, vmax=vmax, rasterized=True)
ax[2].set(xlabel='$X$ [pix]', ylabel='$Y$ [pix]')
fig.colorbar(im, ax=ax[2], fraction=0.046, pad=0.03).set_label('median residual [mag]')
inward(*ax)
savepdf(fig, 'calibration_residuals')

# ---- Fig. 3: colour term with GSPC Johnson B and with the KMTNet passband as reference, same stars, with the residuals of the fits ------
hj = with_gspc(hs0, 'gspc_{}.csv'.format(rec0['stem']))
hj = hj[np.isfinite(hj['b_jkc_mag']) & (hj['b_jkc_mag_err'] < 0.05) & hj['b_jkc_mag'].between(13, 18)]
Aj = design(hj['bp_rp'], hj['X'], hj['Y'], hj['amp'])
cj = hj['bp_rp'].values
xx = np.linspace(*CFG['bp_rp_range'], 100)
fits_j = [(robust_fit(Aj, yv), yv, lab) for yv, lab in (((hj['MAG'] - hj['b_jkc_mag']).values, 'reference: GSPC Johnson $B$'),
                                                        (hj['offset'].values, 'reference: KMTNet passband $B_{\\rm K}$'))]
with plt.rc_context({'axes.labelsize': 11, 'xtick.labelsize': 9, 'ytick.labelsize': 9}):
    fig, ax = plt.subplots(2, 2, figsize=(COL2, 4.1), sharex=True, sharey='row', gridspec_kw=dict(height_ratios=[2.5, 1]), constrained_layout=True)
    for j, (f_, yv, lab) in enumerate(fits_j):
        yy = yv - Aj[:, 1 + K:] @ f_['p'][1 + K:] - f_['p'][0]                 # MAG - reference with the zero point, spatial and amplifier terms removed
        model = lambda c: sum(f_['p'][k] * (c - REF) ** k for k in range(1, K + 1))
        ax[0, j].scatter(cj, yy, s=1.5, color='#8fa8c8', alpha=0.45, rasterized=True)
        ax[0, j].plot(xx, model(xx), color='crimson', lw=1.3, zorder=6)
        b = binned(cj, yy, CFG['color_bins'], min_n=20)
        ax[0, j].errorbar(b[:, 0], b[:, 1], yerr=b[:, 2], fmt='o', color='k', ms=3.5, capsize=2, zorder=5)
        b = binned(cj, yy - model(cj), CFG['color_bins'], min_n=20)
        ax[1, j].errorbar(b[:, 0], b[:, 1], yerr=b[:, 2], fmt='o', color='k', ms=3.5, capsize=2)
        ax[1, j].axhline(0, color='crimson', lw=0.8)
        ax[0, j].axhline(0, color='0.6', lw=0.6, ls=':')
        ax[0, j].text(0.03, 0.95, '{}\n$a_c = {:+.3f}$, $a_{{c^2}} = {:+.3f}$, $\\sigma = {:.3f}$ mag'.format(lab, f_['p'][1], f_['p'][2], f_['sigma']),
                      transform=ax[0, j].transAxes, va='top', fontsize=9, bbox=dict(fc='w', ec='none', alpha=0.85))
        ax[1, j].set_xlabel('Gaia $(G_{\\rm BP}-G_{\\rm RP})$')
    ax[0, 0].set(ylim=(-0.32, 0.32), ylabel='$\\mathrm{MAG} - \\mathrm{MAG}_{\\rm cal}$ [mag]')
    ax[1, 0].set(ylim=(-0.045, 0.045), ylabel='residual [mag]')
    ax[1, 0].set_xlim(cj.min() - 0.1, cj.max() + 0.1)
    inward(*ax.ravel())
savepdf(fig, 'johnson_vs_kmtnet')
print('  {} chip {}: Johnson B a_c = {:+.4f}, a_c2 = {:+.4f}, sigma = {:.4f};  KMTNet passband a_c = {:+.4f}, a_c2 = {:+.4f}, sigma = {:.4f} ({:d} stars)'.format(
      paper_frame, paper_chip, fits_j[0][0]['p'][1], fits_j[0][0]['p'][2], fits_j[0][0]['sigma'], fits_j[1][0]['p'][1], fits_j[1][0]['p'][2],
      fits_j[1][0]['sigma'], int(fits_j[1][0]['keep'].sum())))

# ---- Fig. 4: sky coverage, true chip footprints coloured by the completeness turnover of the deepest frame, with the Roman GBTDS ------
GAIA_HIPS = 'CDS/P/DM/flux-color-Rp-G-Bp/I/355/gaiadr3'                  # Gaia DR3 colour flux map (CDS), via the hips2fits cutout service


def gbtds_union():
    '''One compound path of every GBTDS field outline of both seasons, all with the same orientation, so that a single fill shows their union.'''
    verts, codes = [], []
    for _, _, fp in GBTDS:
        p = fp if np.sum(fp[:-1, 0] * fp[1:, 1] - fp[1:, 0] * fp[:-1, 1]) > 0 else fp[::-1]   # same orientation for every polygon
        verts += list(p) + [p[0]]
        codes += [Path.MOVETO] + [Path.LINETO] * (len(p) - 1) + [Path.CLOSEPOLY]
    return Path(np.array(verts), codes)


def gaia_sky_image(l_range, b_range, pix_deg=0.012):
    '''Gaia DR3 colour flux map on a plate-carree grid in Galactic coordinates covering the ranges, from the CDS hips2fits service
    (cached in gaia_sky_background.npz): returns (rgb image, extent for imshow).'''
    cache = 'gaia_sky_background.npz'
    if os.path.exists(cache):
        z = np.load(cache)
        return z['img'], z['extent']
    nx, ny = int(round(np.ptp(l_range) / pix_deg)), int(round(np.ptp(b_range) / pix_deg))
    hdr = dict(NAXIS1=nx, NAXIS2=ny, CTYPE1='GLON-CAR', CTYPE2='GLAT-CAR', CRVAL1=float(np.mean(l_range)), CRVAL2=0.0, CRPIX1=0.5 * nx + 0.5,
               CRPIX2=0.5 - min(b_range) / pix_deg, CDELT1=-pix_deg, CDELT2=pix_deg, CUNIT1='deg', CUNIT2='deg')
    r = requests.get('https://alasky.cds.unistra.fr/hips-image-services/hips2fits', params=dict(hips=GAIA_HIPS, wcs=json.dumps(hdr), format='png'), timeout=600)
    r.raise_for_status()
    img = plt.imread(io.BytesIO(r.content))
    (l0, b0), (l1, b1) = WCS(hdr).all_pix2world([[0.5, 0.5], [nx + 0.5, ny + 0.5]], 1)       # sky coordinates of the image corners
    extent = np.array([(l0 + 180) % 360 - 180, (l1 + 180) % 360 - 180, b0, b1])
    np.savez_compressed(cache, img=img, extent=extent)
    return img, extent


best = SUM.sort_values('B_turnover').groupby(['object', 'chip']).tail(1)   # deepest frame of every field and chip
norm = plt.Normalize(best['B_turnover'].min(), best['B_turnover'].max())
cmap = plt.cm.viridis
lims = (best['l'].max() + 1.3, best['l'].min() - 1.3), (best['b'].min() - 1.3, best['b'].max() + 1.3)


def coverage_figure(name, background=None):
    '''Chip footprints coloured by the completeness turnover of the deepest frame, the Roman GBTDS footprint, field numbers (fields whose
    centres lie within 0.5 deg -- the half-chip-offset twins 01/41, 02/42, 03/43 -- share one label); on the Gaia sky image the chips use
    a cool colour sequence and the GBTDS is white, to stand apart from the orange background.'''
    on_image = background is not None
    cm = plt.get_cmap('cool' if on_image else 'viridis')
    gb = dict(facecolor='w', edgecolor='none', alpha=0.8) if on_image else dict(facecolor='crimson', edgecolor='none', alpha=0.55)
    with plt.rc_context({'axes.labelsize': 12, 'xtick.labelsize': 10, 'ytick.labelsize': 10, 'legend.fontsize': 9}):
        fig, ax = plt.subplots(figsize=(COL2, 4.4), constrained_layout=True)
        if on_image:
            ax.imshow(background[0], extent=background[1], origin='upper', aspect='equal', interpolation='bilinear', zorder=0)
        for stem, r in best.iterrows():
            ax.add_patch(plt.Polygon(FOOTPRINT[stem], closed=True, facecolor=cm(norm(r['B_turnover'])), alpha=0.6 if on_image else 0.95,
                                     edgecolor='w' if on_image else 'k', lw=0.4 if on_image else 0.3, zorder=3))
        cen = best.groupby('object')[['l', 'b']].mean()
        done = set()
        for obj in cen.index:
            if obj in done:
                continue
            grp = cen.index[(np.hypot(cen['l'] - cen.loc[obj, 'l'], cen['b'] - cen.loc[obj, 'b']) < 0.5).values]
            done |= set(grp)
            ax.text(cen.loc[grp, 'l'].mean(), cen.loc[grp, 'b'].mean(), '/'.join(o.replace('BLG', '') for o in grp), ha='center', va='center',
                    fontsize=8.5, fontweight='bold', color='w' if on_image else 'k',
                    path_effects=[pe.withStroke(linewidth=2.0, foreground='k' if on_image else 'w')], zorder=6)
        ax.add_patch(PathPatch(gbtds_union(), zorder=5, **gb))
        ax.set(xlim=lims[0], ylim=lims[1], xlabel='$\\ell$ [deg]', ylabel='$b$ [deg]')
        ax.set_aspect('equal')
        ax.set_axisbelow(True)
        inward(ax)
        if not on_image:
            ax.grid(ls=':', lw=0.4, color='0.8')
        ax.legend(handles=[Patch(label='Roman GBTDS', **gb)], loc='lower left', frameon=on_image, framealpha=0.6,
                  facecolor='k' if on_image else 'w', labelcolor='w' if on_image else 'k', edgecolor='none')
        cb = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cm), ax=ax, fraction=0.035, pad=0.02)
        cb.set_label('completeness turnover $B_{\\rm K}$', fontsize=11)
        cb.ax.tick_params(labelsize=9)
        savepdf(fig, name)


coverage_figure('coverage')
try:
    coverage_figure('coverage_gaia', background=gaia_sky_image(lims[0][::-1], lims[1]))
except Exception as e:
    print('  coverage on the Gaia sky image skipped: {}'.format(str(e)[:120]))

# ---- Fig. 5: Gaia-KMTNet colour-magnitude diagrams of selected (frame, chip) items, common axes covering >= 99% of each ----------------
def load_cmd(stem):
    '''(B_K, B_K - G) of the matched stars of a calibrated catalogue.'''
    cat = pd.read_csv(stem + '_calibrated_catalog.csv', usecols=['B_K', 'B_minus_G', 'matched'])
    return cat.loc[(cat['matched'] == 1) & np.isfinite(cat['B_minus_G']) & np.isfinite(cat['B_K']), ['B_K', 'B_minus_G']].values


def common_axes(arrays):
    '''(x range, y range) enclosing the central 99% of every array, with a small margin.'''
    return ((min(np.percentile(v[:, 1], 0.5) for v in arrays) - 0.1, max(np.percentile(v[:, 1], 99.5) for v in arrays) + 0.1),
            (min(np.percentile(v[:, 0], 0.5) for v in arrays) - 0.2, max(np.percentile(v[:, 0], 99.5) for v in arrays) + 0.2))


cats = {(fr, ch): load_cmd(rec_of(fr, ch)['stem']) for fr, ch in paper_cmd_items}
xr, yr = common_axes(cats.values())
nc = 2
nr = int(np.ceil(len(cats) / nc))
fig, ax = plt.subplots(nr, nc, figsize=(COL2, 3.4 * nr), squeeze=False, constrained_layout=True)
for k, ((fr, ch), v) in enumerate(cats.items()):
    axx = ax[k // nc, k % nc]
    r = rec_of(fr, ch)
    axx.hist2d(v[:, 1], v[:, 0], bins=140, range=[xr, yr], norm=LogNorm(), cmap='magma', rasterized=True)
    axx.set(xlim=xr, ylim=(yr[1], yr[0]), xlabel='$B_{\\rm K} - G$', ylabel='$B_{\\rm K}$')
    axx.tick_params(direction='in', top=True, right=True, color='w')
    axx.text(0.97, 0.96, '{} chip {}\n$(\\ell, b) = ({:+.2f}^\\circ, {:+.2f}^\\circ)$\n$N = {:,d}$'.format(r['object'], ch, r['l'], r['b'], len(v)),
             transform=axx.transAxes, fontsize=9, ha='right', va='top', bbox=dict(fc='w', ec='none', alpha=0.85))
for k in range(len(cats), nr * nc):
    ax[k // nc, k % nc].set_visible(False)
savepdf(fig, 'cmds')

# ---- Fig. 6: precision and depth of the same items -------------------------------------------------------------------------------------
fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.8), constrained_layout=True)
for k, (fr, ch) in enumerate(paper_cmd_items):
    r = rec_of(fr, ch)
    cv = np.array(r['depth']['merr_vs_B'])
    lf = np.array(r['depth']['lf_B'], float)
    ax[0].plot(cv[:, 0], cv[:, 1], '-', lw=1.2, color='C{:d}'.format(k), label='{} {} ({:+.1f}, {:+.1f})'.format(r['object'], ch, r['l'], r['b']))
    ax[1].plot(r['depth']['lf_mids'], lf / lf.max(), '-', lw=1.2, color='C{:d}'.format(k))
ax[0].set(yscale='log', xlabel='$B_{\\rm K}$', ylabel='median DAOPHOT error [mag]')
ax[0].legend(frameon=False, fontsize=7)
ax[1].set(xlabel='$B_{\\rm K}$', ylabel='normalised counts per 0.2 mag', xlim=(13, 23), yscale='log', ylim=(1e-3, 1.5))
inward(*ax)
savepdf(fig, 'precision_depth')

# ---- Vega zero point of the B_K system -----------------------------------------------------------------------------------------------------
if CFG['vega_file'] and os.path.exists(CFG['vega_file']):
    vt = Table.read(CFG['vega_file'])
    fv = np.interp(GRID, np.asarray(vt['WAVELENGTH'], float) / 10.0, np.asarray(vt['FLUX'], float) / 100.0)[None, :]   # A -> nm, erg/s/cm2/A -> W/m2/nm
    print('AB magnitude of Vega through the system passband: {:+.4f}'.format(float(ab_mag(fv, fv * 0, S_SYS)[0][0])))
else:
    print('Vega zero point: no CALSPEC file (CFG vega_file).')
plt.rcParams.update(SCREEN_RC)

## 7b. Depth relative to Gaia
For a few fields spanning the range of stellar density, a Hess diagram with the faint limits of the published XP spectra
($G = 17.65$) and of the Gaia catalog used ($G = 20.5$) drawn as lines, and the luminosity functions in $B_{\rm K}$ of the detections
split by what Gaia has for them (a counterpart, a $G_{\rm BP}-G_{\rm RP}$ colour, an XP spectrum). One PDF per field in `figs/`, each
with its LaTeX figure environment printed; `depth_items` selects (field, chip), the deepest frame of each is used.

In [ ]:
depth_items = [('BLG16', 'K'), ('BLG01', 'T'), ('BLG52', 'T'), ('BLG31', 'T'), ('BLG33', 'T')]   # (field, chip): sparse, medium, heavily reddened, dense, densest
G_XP, G_CAT = 17.65, CFG['gaia_gmax']                                       # faint limit of the published XP spectra; of the Gaia catalogue used


def deepest_record(obj, chip):
    '''The record of the deepest frame (most matched stars) of a field and chip.'''
    return max((r for r in records.values() if r['object'] == obj and r['chip'] == chip), key=lambda r: r['match']['n_matched'])


plt.rcParams.update(PAPER_RC)
for obj, ch in depth_items:
    r = deepest_record(obj, ch)
    cat = pd.read_csv(r['stem'] + '_calibrated_catalog.csv', usecols=['B_K', 'B_minus_G', 'matched', 'bp_rp', 'phot_g_mean_mag', 'has_xp_continuous'])
    m = cat[cat['matched'] == 1]
    sets = [('all DAOPHOT detections', cat['B_K'], 'k'), ('with a Gaia counterpart', m['B_K'], 'C0'),
            ('with a $G_{\\rm BP}-G_{\\rm RP}$ colour', m.loc[np.isfinite(m['bp_rp']), 'B_K'], 'C1'), ('with an XP spectrum', m.loc[m['has_xp_continuous'] == 1, 'B_K'], 'C2')]
    bins = np.arange(np.floor(cat['B_K'].min()), 24.01, 0.2)
    mids = 0.5 * (bins[1:] + bins[:-1])
    fig, ax = plt.subplots(1, 2, figsize=(COL2, 3.0), constrained_layout=True)
    # left: Hess diagram of the matched stars; a constant G is a diagonal B_K = G + (B_K - G)
    xr_, yr_ = common_axes([m[['B_K', 'B_minus_G']].values])
    ax[0].hist2d(m['B_minus_G'].values, m['B_K'].values, bins=130, range=[xr_, yr_], norm=LogNorm(), cmap='magma', rasterized=True)
    xx = np.array(xr_)
    ax[0].plot(xx, G_XP + xx, 'k--', lw=1.0, label='$G = {:.2f}$: limit of the XP spectra'.format(G_XP))
    ax[0].plot(xx, G_CAT + xx, 'k:', lw=1.2, label='$G = {:.1f}$: limit of the Gaia catalog used'.format(G_CAT))
    ax[0].set(xlim=xr_, ylim=(yr_[1], yr_[0]), xlabel='$B_{\\rm K} - G$', ylabel='$B_{\\rm K}$')
    ax[0].legend(loc='lower left', fontsize=6.5, frameon=True, framealpha=0.85, edgecolor='none')
    ax[0].text(0.97, 0.96, '{} chip {}\n$(\\ell, b) = ({:+.1f}^\\circ, {:+.1f}^\\circ)$'.format(obj, ch, r['l'], r['b']), transform=ax[0].transAxes, ha='right', va='top', fontsize=8, bbox=dict(fc='w', ec='none', alpha=0.85))
    # right: luminosity functions
    for lab, v, col in sets:
        h = np.histogram(v, bins)[0].astype(float)
        h[h == 0] = np.nan
        ax[1].plot(mids, h, '-', color=col, lw=1.3, label=lab)
    ax[1].set(yscale='log', xlabel='$B_{\\rm K}$', ylabel='stars per 0.2 mag', xlim=(12, 23))
    ax[1].legend(loc='upper left', fontsize=6.5, frameon=False)
    inward(*ax)
    savepdf(fig, 'depth_gaia_{}_{}'.format(obj, ch))
    n_all, n_m, n_c, n_x = (len(v) for _, v, _ in sets)
    print('\n'.join([r'\begin{figure*}', r'\centering', r'\includegraphics[width=\textwidth]{Figures/paper_depth_gaia_%s_%s.pdf}' % (obj, ch),
                     r'\caption{Depth of the catalog relative to \gaia, for field %s, chip %s. Left: Hess diagram of the detections with a \gaia\ counterpart; '
                     r'because $B_{\rm K} = G + (B_{\rm K} - G)$, a fixed $G$ is a diagonal line, and the dashed and dotted lines mark $G = %.2f$, the faint limit '
                     r'of the published XP spectra, and $G = %.1f$, the limit to which \gaia\ sources were used (\gaia\ itself reaches $G \approx 21$): stars between the '
                     r'two lines have \gaia\ photometry but no spectrum. Right: luminosity functions in $B_{\rm K}$ of all %s DAOPHOT detections and of those with a '
                     r'\gaia\ counterpart (%s), with a $G_{\rm BP}-G_{\rm RP}$ colour (%s) and with an XP spectrum (%s).}' % (obj, ch, G_XP, G_CAT, format(n_all, ','), format(n_m, ','), format(n_c, ','), format(n_x, ',')),
                     r'\label{fig:depth_gaia}', r'\end{figure*}', '']))
plt.rcParams.update(SCREEN_RC)

## 8. All colour–magnitude diagrams
Hess diagrams of every calibrated chip-frame on common axes (the union of each panel's central 99%), in two arrangements:
`figs/cmds_all_<chip>.png/.pdf` (one figure per chip, every frame; panels whose chip overlaps the Roman GBTDS footprint are grey) and
`figs/cmds_fields_<page>.pdf` (one panel per field, the four chips of its deepest frame combined, `cmd_fields_per_page` fields per
page; fields overlapping the GBTDS are grey). The LaTeX for the field figure is written to `figs/cmds_fields.tex`.

In [ ]:
cmd_all_ncols, cmd_bins = 8, 110                     # per-chip figures: panels per row, Hess bins
cmd_fields_per_page = 30                             # per-field figure: fields per page (29 fields -> one page)
plt.rcParams.update(PAPER_RC)
recs_all = sorted(records.values(), key=lambda r: (r['chip'], r['frame']))
cats_all = {r['stem']: load_cmd(r['stem']) for r in recs_all}
xr_all, yr_all = common_axes(cats_all.values())
print('common axes: B_K - G in {:.2f}..{:.2f}, B_K in {:.2f}..{:.2f}; {:d} chip-frames'.format(*xr_all, *yr_all, len(recs_all)))


def hess(ax, v, bins, shaded=False):
    '''Hess diagram of (B_K, B_K - G) rows v on the common axes; shaded = grey-blue background (overlaps the Roman GBTDS footprint).'''
    if shaded:
        ax.set_facecolor('#aab6c8')
    ax.hist2d(v[:, 1], v[:, 0], bins=bins, range=[xr_all, yr_all], norm=LogNorm(), cmap='magma', rasterized=True)
    ax.set(xlim=xr_all, ylim=(yr_all[1], yr_all[0]))


# ---- one figure per chip, every frame --------------------------------------------------------------------------------------------------
for ch in CFG['chips']:
    grp = [r for r in recs_all if r['chip'] == ch]
    nc = cmd_all_ncols
    nr = int(np.ceil(len(grp) / nc))
    fig, axes = plt.subplots(nr, nc, figsize=(2.3 * nc, 2.5 * nr), squeeze=False, constrained_layout=True)
    for k, r in enumerate(grp):
        ax = axes[k // nc, k % nc]
        hess(ax, cats_all[r['stem']], cmd_bins, SUM.loc[r['stem'], 'gbtds'])
        ax.tick_params(labelsize=7, direction='in', top=True, right=True, length=2)
        ax.text(0.97, 0.96, '{} {}\n({:+.1f}, {:+.1f})  N={:,d}'.format(r['object'], r['chip'], r['l'], r['b'], len(cats_all[r['stem']])),
                transform=ax.transAxes, fontsize=7, ha='right', va='top', bbox=dict(fc='w', ec='none', alpha=0.8))
        if k % nc == 0:
            ax.set_ylabel('$B_{\\rm K}$')
        if k // nc == nr - 1:
            ax.set_xlabel('$B_{\\rm K} - G$')
    for k in range(len(grp), nr * nc):
        axes[k // nc, k % nc].set_visible(False)
    fig.suptitle('Gaia-KMTNet Hess diagrams, chip {} ({:d} frames)'.format(ch, len(grp)))
    for ext in ('png', 'pdf'):
        fig.savefig('figs/cmds_all_{}.{}'.format(ch, ext), dpi=110 if ext == 'png' else None, bbox_inches='tight')
    plt.show()
    print('  wrote figs/cmds_all_{}.png / .pdf'.format(ch))

# ---- one panel per field (deepest frame, four chips combined), paginated ----------------------------------------------------------------
by_field = {}                                                                 # object -> frame -> chip -> record
for r in records.values():
    by_field.setdefault(r['object'], {}).setdefault(r['frame'], {})[r['chip']] = r
blocks = []                                                                   # (object, deepest frame, {chip: record})
for obj in sorted(by_field):
    frames = by_field[obj]
    fr = max(frames, key=lambda f: sum(x['match']['n_matched'] for x in frames[f].values()))
    blocks.append((obj, fr, frames[fr]))


def field_page(page, fname, ncol=5):
    '''One page of single-panel field CMDs (shared axes, tick labels on the outer panels only), written to fname.pdf/.png.'''
    nrow = int(np.ceil(len(page) / ncol))
    H = 0.3 + 1.32 * nrow
    fig, axes = plt.subplots(nrow, ncol, figsize=(COL2, H), sharex=True, sharey=True, squeeze=False)
    fig.subplots_adjust(left=0.075, right=0.995, bottom=0.42 / H, top=0.995, wspace=0.03, hspace=0.045)
    for k, (obj, fr, recs) in enumerate(page):
        ax = axes[k // ncol, k % ncol]
        v = np.vstack([cats_all[x['stem']] for x in recs.values()])
        shaded = any(SUM.loc[x['stem'], 'gbtds'] for x in recs.values())
        hess(ax, v, 110, shaded)
        ax.set(xticks=xt, yticks=yt)
        ax.tick_params(direction='in', top=True, right=True, length=2, labelsize=7, color='0.4')
        l_c, b_c = np.mean([x['l'] for x in recs.values()]), np.mean([x['b'] for x in recs.values()])
        ax.text(0.96, 0.96, '({:+.1f}, {:+.1f})\n$N$ = {:,d}'.format(l_c, b_c, len(v)), transform=ax.transAxes, ha='right', va='top',
                fontsize=8, linespacing=1.3, bbox=dict(fc='w', ec='none', alpha=0.85, pad=1.5))
    for k in range(len(page), nrow * ncol):
        axes[k // ncol, k % ncol].set_visible(False)
    for j in range(ncol):                                                       # x tick labels on the lowest visible panel of every column
        vis = [axes[i, j] for i in range(nrow) if axes[i, j].get_visible()]
        if vis:
            vis[-1].tick_params(labelbottom=True)
    fig.supxlabel('$B_{\\rm K} - G$', fontsize=10, y=0.006)
    fig.supylabel('$B_{\\rm K}$', fontsize=10, x=0.012)
    fig.savefig(fname + '.pdf', bbox_inches='tight', pad_inches=0.02)
    fig.savefig(fname + '.png', dpi=150, bbox_inches='tight')
    plt.show()


xt, yt = np.arange(0, xr_all[1] + 1e-6, 2), np.arange(12, yr_all[1] + 1e-6, 2)
pages = [blocks[i:i + cmd_fields_per_page] for i in range(0, len(blocks), cmd_fields_per_page)]
for ip, page in enumerate(pages, 1):
    field_page(page, 'figs/cmds_fields_{:d}'.format(ip))
    print('  wrote figs/cmds_fields_{:d}.pdf / .png ({:d} fields)'.format(ip, len(page)))

# ---- LaTeX of the field figure ------------------------------------------------------------------------------------------------------------
n_sh = sum(any(SUM.loc[x['stem'], 'gbtds'] for x in recs.values()) for _, _, recs in blocks)
cap = (r'Gaia--KMTNet Hess diagrams of the %d fields, one panel per field combining the four chips of its deepest frame; all panels share the axes shown. '
       r'Each panel is labelled with the Galactic coordinates of the field centre and the number of matched stars (fields are identified in Figure~\ref{fig:coverage}). '
       r'Fields with a grey-blue background (%d) overlap the footprint of the Roman Galactic Bulge Time Domain Survey.' % (len(blocks), n_sh))
tex = '\n'.join([r'\begin{figure*}', r'\centering', r'\includegraphics[width=\textwidth]{figs/cmds_fields_1.pdf}', r'\caption{%s\label{fig:cmds_fields}}' % cap, r'\end{figure*}']
                + sum([[r'', r'\begin{figure*}', r'\centering', r'\figurenum{\ref{fig:cmds_fields}}', r'\includegraphics[width=\textwidth]{figs/cmds_fields_%d.pdf}' % ip,
                        r'\caption{Continued.}', r'\end{figure*}'] for ip in range(2, len(pages) + 1)], []))
open('figs/cmds_fields.tex', 'w').write(tex + '\n')
print(tex)
plt.rcParams.update(SCREEN_RC)

## 9. LaTeX table of the calibration coefficients
Median over all chip-frames of the program and the frame-to-frame scatter of every coefficient, written to `coefficients_table.tex`.
`table_scatter` chooses how the scatter is measured: `'within chips'` (the standard deviation between the frames of a chip, pooled
over the four chips -- the genuine frame-to-frame variation) or `'all chip-frames'` (the standard deviation over all 229 records, which
for the surface and amplifier terms also contains the differences between chips).

In [ ]:
table_scatter = 'within chips'                     # 'within chips' or 'all chip-frames' (see above)

SYMBOL = {'a': r'$a_0$', 'c1': r'$a_c$', 'c2': r'$a_{c^2}$', 's_y': r'$a_y$', 's_y2': r'$a_{y^2}$', 's_xy': r'$a_{xy}$', 's_x2y': r'$a_{x^2y}$',
          's_xy2': r'$a_{xy^2}$', 's_x': r'$a_x$', 's_x2': r'$a_{x^2}$', 's_y3': r'$a_{y^3}$'}
DESCR = {'a': r'Zero point: chip-averaged ${\rm MAG} - B_{\rm K}$ at $(G_{\rm BP}-G_{\rm RP}) = %.1f$, including the atmospheric extinction at the airmass of the frame' % REF,
         'c1': r'Colour term, linear in $\Delta c = (G_{\rm BP}-G_{\rm RP}) - %.1f$ (mag per mag)' % REF,
         'c2': r'Colour term, quadratic in $\Delta c$ (mag per mag$^2$)',
         's_y': r'Gradient along $y$ (bottom-to-top change $2a_y$)', 's_y2': r'Curvature along $y$ (coefficient of $y^2 - 1/3$)',
         's_xy': r'Change of the $y$-gradient with $x$', 's_x2y': r'Quadratic-in-$x$ modulation of the $y$-gradient',
         's_xy2': r'Linear-in-$x$ modulation of the $y$-curvature', 's_x': r'Gradient along $x$', 's_x2': r'Curvature along $x$', 's_y3': r'Cubic term in $y$'}
ROWS = [(SYMBOL.get(n_, n_), DESCR.get(n_, n_), n_) for n_ in NAMES if not n_.startswith('g_')]
ROWS += [(r'$a_{{\rm amp},%d}$' % j, 'Offset of amplifier %d%s' % (j, r' (derived: $-\sum_{j \geq 2} a_{{\rm amp},j}$)' if j == 1 else ''), 'amp%d' % j)
         for j in range(1, N_AMP + 1)]


def scatter_of(col):
    '''Frame-to-frame scatter of one coefficient, as chosen by table_scatter.'''
    if table_scatter == 'within chips':
        return float(np.sqrt(np.mean([SUM.loc[SUM['chip'] == c, col].var(ddof=1) for c in CFG['chips'] if (SUM['chip'] == c).sum() > 1])))
    return float(SUM[col].std(ddof=1))


def fmt(med, sc):
    '''Median and scatter with the number of decimals set by the scatter (two significant digits of the scatter).'''
    d = min(4, max(0, 1 - int(np.floor(np.log10(sc))))) if sc > 0 else 3
    return '${:+.{d}f}$'.format(med, d=d), '{:.{d}f}'.format(sc, d=d)


err_formal = float(np.median([np.median(r['err_formal']) for r in records.values()]))
chips_tab = [c for c in CFG['chips'] if (SUM['chip'] == c).any()]
per_chip = lambda f: ', '.join(f(SUM[SUM['chip'] == c]) for c in chips_tab)
scatter_text = ('the standard deviation between the frames of a chip, pooled over the four chips' if table_scatter == 'within chips'
                else 'the standard deviation over all chip-frames, which for the surface and amplifier terms includes the differences between chips')
cap = (r'Coefficients of Equation~\ref{eq:model}, all of which are fitted independently for every frame and chip: the zero point follows the transparency, '
       r'airmass and exposure time; the colour coefficients absorb the residual of the passband model, the colour dependence of the atmospheric extinction and '
       r'the crowding-dependent blending of the field; and the surface and amplifier terms absorb the frame-specific effects of the seeing, the DAOPHOT PSF model, '
       r'the stellar density and any residual error of the flat field. We list the median over the %d chip-frames of the program and the frame-to-frame scatter '
       r'(%s); the formal $1\sigma$ errors of the single-frame fits ($\approx %.4f$~mag) are much smaller than the scatter, which therefore measures the genuine '
       r'frame-to-frame variation. Values are in magnitudes unless noted; chip coordinates are normalized to $[-1,1]$; amplifier offsets sum to zero. '
       r'\label{tab:coefficients}' % (len(SUM), scatter_text, err_formal))
lines = [r'\begin{deluxetable*}{lp{8.6cm}cc}', r'\tablewidth{0pt}', r'\tablecaption{%s}' % cap,
         r'\tablehead{\colhead{Coefficient} & \colhead{Meaning} & \colhead{Median} & \colhead{Frame-to-frame scatter}}', r'\startdata']
for sym, descr, col in ROWS:
    med, sc = fmt(float(SUM[col].median()), scatter_of(col))
    lines.append(r'%s & %s & %s & %s \\' % (sym, descr, med, sc))
lines += [r'\enddata',
          r'\tablecomments{The colour coefficients grow with stellar density (Section~\ref{sec:colorterm}). Chips %s: %s chip-frames; median $N_{\rm cal}$ = %s; '
          r'median $\sigma_{\rm MAD}$ = %s~mag.}' % (', '.join(chips_tab), per_chip(lambda g: str(len(g))), per_chip(lambda g: '{:.0f}'.format(g['n_calib'].median())),
                                                   per_chip(lambda g: '{:.3f}'.format(g['sigma_mad'].median()))),
          r'\end{deluxetable*}']
open('coefficients_table.tex', 'w').write('\n'.join(lines) + '\n')
print('\n'.join(lines))
print('\n(written to coefficients_table.tex)')